# Continuous connector VLM

Notebook supplied by Muhammad Bin Tariq; see the repository provenance for the accompanying script implementation. Saved outputs and runtime metadata have been cleared for publication; recorded results and their original cell indices are documented in the project README. The original source notebook is unchanged.

This is an experimental workflow, not a claim of a newly reproduced run. Review configuration and GPU/model access requirements before execution.


Runtime sanity check

In [ ]:
import os
import sys
import platform
import subprocess
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU capability:", torch.cuda.get_device_capability(0))
    print("Total VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("WARNING: CUDA is not available. Please enable GPU runtime.")

Install required packages

In [ ]:
!pip -q install \
    transformers \
    datasets \
    accelerate \
    peft \
    torchvision \
    scikit-learn \
    matplotlib \
    tqdm \
    umap-learn

Imports, folders, reproducibility

In [ ]:
import os
import math
import time
import json
import random
import shutil
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# ----------------------------
# Reproducibility
# ----------------------------

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic algorithms can slow training and sometimes conflict with CUDA ops.
    # For this assignment, reproducibility + practical speed is the better balance.
    torch.backends.cudnn.benchmark = True

seed_everything(42)

# ----------------------------
# TF32 on A100
# ----------------------------

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ----------------------------
# Project folders
# ----------------------------

PROJECT_DIR = "/content/pa3_vlm"
WEIGHTS_DIR = os.path.join(PROJECT_DIR, "weights")
OUTPUTS_DIR = os.path.join(PROJECT_DIR, "outputs")
CACHE_DIR = os.path.join(PROJECT_DIR, "cache")

for d in [PROJECT_DIR, WEIGHTS_DIR, OUTPUTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Weights directory:", WEIGHTS_DIR)
print("Outputs directory:", OUTPUTS_DIR)
print("Cache directory:", CACHE_DIR)

Global config

In [ ]:
@dataclass
class Config:
    seed: int = 42

    # Models
    lm_name: str = "HuggingFaceTB/SmolLM2-360M-Instruct"
    expected_lm_hidden_size: int = 960
    expected_lm_vocab_size: int = 49152

    # Training precision
    dtype: torch.dtype = torch.float16

    # Alpaca replay
    alpaca_name: str = "tatsu-lab/alpaca"
    alpaca_replay_size: int = 1000
    ppl_eval_max_examples: int = 100  # use 100 for fast baseline; later can increase to 1000

    # Tokenization
    max_text_length: int = 256

    # LoRA config required by manual
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_targets: Tuple[str, ...] = ("q_proj", "k_proj", "v_proj", "o_proj")

cfg = Config()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(cfg)
print("Device:", DEVICE)

Utility functions

In [ ]:
def get_gpu_memory_gb() -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"allocated": 0.0, "reserved": 0.0, "max_allocated": 0.0}

    return {
        "allocated": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "max_allocated": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }


def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def count_parameters(model: nn.Module) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return {
        "total": total,
        "trainable": trainable,
        "frozen": frozen,
    }


def print_parameter_summary(model: nn.Module, name: str = "model"):
    stats = count_parameters(model)
    pct = 100.0 * stats["trainable"] / max(stats["total"], 1)

    print(f"\nParameter summary: {name}")
    print("-" * 60)
    print(f"Total params:     {stats['total']:,}")
    print(f"Trainable params: {stats['trainable']:,}")
    print(f"Frozen params:    {stats['frozen']:,}")
    print(f"Trainable %:      {pct:.4f}%")
    print("-" * 60)


def set_requires_grad(model: nn.Module, requires_grad: bool):
    for p in model.parameters():
        p.requires_grad = requires_grad


def safe_decode(tokenizer, token_ids):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().tolist()
    return tokenizer.decode(token_ids, skip_special_tokens=False)


def format_instruction_example(example: Dict[str, Any]) -> str:
    """
    Converts an Alpaca example into a simple instruction-following text.
    We use this for language replay perplexity and later LM replay batches.

    Alpaca fields are usually:
    instruction, input, output
    """
    instruction = example.get("instruction", "").strip()
    input_text = example.get("input", "").strip()
    output = example.get("output", "").strip()

    if input_text:
        text = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Input:\n"
            f"{input_text}\n\n"
            "### Response:\n"
            f"{output}"
        )
    else:
        text = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
            f"{output}"
        )

    return text


@torch.no_grad()
def compute_perplexity(
    model: nn.Module,
    tokenizer,
    texts: List[str],
    device: torch.device,
    batch_size: int = 4,
    max_length: int = 256,
    desc: str = "Computing PPL",
) -> float:
    """
    Standard causal LM perplexity:
        PPL = exp(total negative log likelihood / total number of valid labels)

    This computes loss over the full text sequence.
    For replay/forgetting, all examples are pure text.
    """
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )

        # outputs.loss is averaged over non-ignored labels
        valid_tokens = (labels != -100).sum().item()
        total_nll += outputs.loss.item() * valid_tokens
        total_tokens += valid_tokens

    avg_nll = total_nll / max(total_tokens, 1)
    ppl = math.exp(avg_nll)

    return ppl

Load tokenizer and SmolLM2

In [ ]:

reset_peak_memory()

print("Loading tokenizer:", cfg.lm_name)
tokenizer = AutoTokenizer.from_pretrained(cfg.lm_name)

# Decoder-only models should use left padding for batched generation.
# For loss computation, either side can work, but we keep left padding consistently.
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    print("Tokenizer has no pad_token. Setting pad_token = eos_token.")
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab size:", len(tokenizer))
print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("bos_token:", tokenizer.bos_token, tokenizer.bos_token_id)

print("\nLoading LM:", cfg.lm_name)
lm = AutoModelForCausalLM.from_pretrained(
    cfg.lm_name,
    torch_dtype=cfg.dtype,
    device_map=None,
)

lm.to(DEVICE)
lm.eval()

# Disable cache during training/loss computation.
lm.config.use_cache = False

print("\nLM loaded.")
print("LM dtype of first parameter:", next(lm.parameters()).dtype)

hidden_size = lm.config.hidden_size
vocab_size = lm.config.vocab_size

print("LM hidden size:", hidden_size)
print("LM config vocab size:", vocab_size)
print("Tokenizer length:", len(tokenizer))

assert hidden_size == cfg.expected_lm_hidden_size, f"Expected hidden size {cfg.expected_lm_hidden_size}, got {hidden_size}"
assert vocab_size == cfg.expected_lm_vocab_size, f"Expected vocab size {cfg.expected_lm_vocab_size}, got {vocab_size}"

print_parameter_summary(lm, "SmolLM2 before LoRA")
print("GPU memory GB:", get_gpu_memory_gb())

Load Alpaca replay subset

In [ ]:

print("Loading Alpaca:", cfg.alpaca_name)
alpaca = load_dataset(cfg.alpaca_name, split="train")

print(alpaca)
print("Columns:", alpaca.column_names)
print("Total examples:", len(alpaca))

# Shuffle deterministically and select 1,000 examples as required.
alpaca_subset = alpaca.shuffle(seed=cfg.seed).select(range(cfg.alpaca_replay_size))

alpaca_texts = [format_instruction_example(ex) for ex in alpaca_subset]

print("Replay examples:", len(alpaca_texts))
print("\nExample replay text:")
print("-" * 80)
print(alpaca_texts[0][:1000])
print("-" * 80)

# Small sanity tokenization check
tok = tokenizer(
    alpaca_texts[:2],
    padding=True,
    truncation=True,
    max_length=cfg.max_text_length,
    return_tensors="pt",
)

print("Tokenized input_ids shape:", tok["input_ids"].shape)
print("Attention mask shape:", tok["attention_mask"].shape)
print("First decoded sample:")
print(safe_decode(tokenizer, tok["input_ids"][0]))

Compute baseline language perplexity PPL0

In [ ]:
# For speed, first compute on 100 examples.
# Later, if needed, we can compute on all 1,000 examples.
ppl0_texts = alpaca_texts[:cfg.ppl_eval_max_examples]

print(f"Computing PPL0 on {len(ppl0_texts)} Alpaca replay examples...")
reset_peak_memory()

PPL0 = compute_perplexity(
    model=lm,
    tokenizer=tokenizer,
    texts=ppl0_texts,
    device=DEVICE,
    batch_size=4,
    max_length=cfg.max_text_length,
    desc="PPL0",
)

print("\nBaseline PPL0:", PPL0)
print("GPU memory GB:", get_gpu_memory_gb())

# Save value for later forgetting ratio R = PPL_fine / PPL0
ppl0_path = os.path.join(OUTPUTS_DIR, "ppl0.json")
with open(ppl0_path, "w") as f:
    json.dump(
        {
            "PPL0": PPL0,
            "num_examples": len(ppl0_texts),
            "max_length": cfg.max_text_length,
            "model": cfg.lm_name,
        },
        f,
        indent=2,
    )

print("Saved PPL0 to:", ppl0_path)

Part A config and CIFAR-10 metadata

In [ ]:
from torchvision.datasets import CIFAR10
from transformers import CLIPImageProcessor, CLIPVisionModel
from PIL import Image

@dataclass
class PartAConfig:
    clip_name: str = "openai/clip-vit-base-patch32"

    # Dataset sizes required by manual
    train_per_class: int = 1000
    test_per_class: int = 200

    # CLIP / connector dimensions
    clip_patch_tokens: int = 49
    clip_hidden_size: int = 768
    lm_hidden_size: int = 960

    # Cache
    cache_clip_features: bool = True
    clip_feature_dtype: torch.dtype = torch.float16

partA_cfg = PartAConfig()

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

CIFAR10_LABEL_TO_NAME = {i: name for i, name in enumerate(CIFAR10_CLASSES)}
CIFAR10_NAME_TO_LABEL = {name: i for i, name in enumerate(CIFAR10_CLASSES)}

VEHICLE_CLASSES = {"airplane", "automobile", "ship", "truck"}
LIVING_CLASSES = {"bird", "cat", "deer", "dog", "frog", "horse"}
ANIMAL_CLASSES = {"bird", "cat", "deer", "dog", "frog", "horse"}
CAN_FLY_CLASSES = {"airplane", "bird"}

print("Part A config:", partA_cfg)
print("CIFAR-10 classes:", CIFAR10_CLASSES)

Load CIFAR-10 and build stratified subsets

In [ ]:
DATA_DIR = os.path.join(PROJECT_DIR, "data")

cifar_train_full = CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=None,
)

cifar_test_full = CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=None,
)

print("Full CIFAR train:", len(cifar_train_full))
print("Full CIFAR test:", len(cifar_test_full))

def stratified_cifar_indices(targets: List[int], per_class: int, seed: int = 42) -> List[int]:
    """
    Deterministically sample exactly per_class examples from each CIFAR-10 class.
    """
    rng = np.random.default_rng(seed)
    targets = np.array(targets)

    selected = []

    for cls_id in range(10):
        cls_indices = np.where(targets == cls_id)[0]
        rng.shuffle(cls_indices)

        if len(cls_indices) < per_class:
            raise ValueError(f"Class {cls_id} has only {len(cls_indices)} examples, requested {per_class}")

        selected.extend(cls_indices[:per_class].tolist())

    rng.shuffle(selected)
    return selected

train_indices_A = stratified_cifar_indices(
    cifar_train_full.targets,
    per_class=partA_cfg.train_per_class,
    seed=cfg.seed,
)

test_indices_A = stratified_cifar_indices(
    cifar_test_full.targets,
    per_class=partA_cfg.test_per_class,
    seed=cfg.seed,
)

print("Part A train subset size:", len(train_indices_A))
print("Part A test subset size:", len(test_indices_A))

def count_classes(dataset, indices):
    counts = {name: 0 for name in CIFAR10_CLASSES}
    for idx in indices:
        _, label = dataset[idx]
        counts[CIFAR10_LABEL_TO_NAME[label]] += 1
    return counts

train_counts_A = count_classes(cifar_train_full, train_indices_A)
test_counts_A = count_classes(cifar_test_full, test_indices_A)

print("\nTrain class counts:")
print(train_counts_A)

print("\nTest class counts:")
print(test_counts_A)

assert all(v == partA_cfg.train_per_class for v in train_counts_A.values())
assert all(v == partA_cfg.test_per_class for v in test_counts_A.values())

print("\nStratified subset checks passed.")

Visual sanity check for CIFAR subset

In [ ]:
def show_cifar_grid(dataset, indices, n_per_class: int = 5):
    """
    Show n_per_class examples from each CIFAR-10 class.
    """
    class_to_indices = {cls_id: [] for cls_id in range(10)}

    for idx in indices:
        _, label = dataset[idx]
        if len(class_to_indices[label]) < n_per_class:
            class_to_indices[label].append(idx)

    rows = 10
    cols = n_per_class

    plt.figure(figsize=(cols * 2, rows * 1.8))

    plot_idx = 1
    for cls_id in range(10):
        for idx in class_to_indices[cls_id]:
            image, label = dataset[idx]
            plt.subplot(rows, cols, plot_idx)
            plt.imshow(image)
            plt.axis("off")
            if plot_idx % cols == 1:
                plt.title(CIFAR10_LABEL_TO_NAME[label], fontsize=10)
            plot_idx += 1

    plt.tight_layout()
    plt.show()

show_cifar_grid(cifar_train_full, train_indices_A, n_per_class=5)

Caption and VQA generation

In [ ]:
CAPTION_TEMPLATES = [
    "A photo of a {cls}.",
    "This image shows a {cls}.",
    "The object in the image is a {cls}.",
    "A small image of a {cls}.",
    "There is a {cls} in the picture.",
    "This is a CIFAR-10 image of a {cls}.",
]

def make_caption(label: int, sample_id: int) -> str:
    cls = CIFAR10_LABEL_TO_NAME[label]
    template = CAPTION_TEMPLATES[sample_id % len(CAPTION_TEMPLATES)]
    return template.format(cls=cls)


def get_vehicle_or_living_answer(cls_name: str) -> str:
    if cls_name in VEHICLE_CLASSES:
        return "vehicle"
    return "living"


def get_can_fly_answer(cls_name: str) -> str:
    return "yes" if cls_name in CAN_FLY_CLASSES else "no"


def get_is_animal_answer(cls_name: str) -> str:
    return "yes" if cls_name in ANIMAL_CLASSES else "no"


def negative_class_for_label(label: int, sample_id: int) -> str:
    """
    Deterministic negative class for balanced yes/no questions.
    Ensures target class != true class.
    """
    offset = 1 + (sample_id % 9)
    neg_label = (label + offset) % 10
    return CIFAR10_LABEL_TO_NAME[neg_label]


def make_vqa_pairs_for_image(label: int, sample_id: int) -> List[Dict[str, str]]:
    """
    Five VQA templates required by the manual.

    Important nuance:
    The manual lists "Is there a {class}?" as a yes/no binary question.
    To make this actually binary rather than always "yes", we alternate between
    the true class and a deterministic wrong class.
    """
    cls = CIFAR10_LABEL_TO_NAME[label]

    # Balanced yes/no class-presence question
    if sample_id % 2 == 0:
        queried_cls = cls
        presence_answer = "yes"
    else:
        queried_cls = negative_class_for_label(label, sample_id)
        presence_answer = "no"

    pairs = [
        {
            "template_id": "object_recognition",
            "question": "What object is shown?",
            "answer": cls,
        },
        {
            "template_id": "class_presence",
            "question": f"Is there a {queried_cls}?",
            "answer": presence_answer,
        },
        {
            "template_id": "vehicle_or_living",
            "question": "Vehicle or living thing?",
            "answer": get_vehicle_or_living_answer(cls),
        },
        {
            "template_id": "can_fly",
            "question": "Can it fly?",
            "answer": get_can_fly_answer(cls),
        },
        {
            "template_id": "is_animal",
            "question": "Is this an animal?",
            "answer": get_is_animal_answer(cls),
        },
    ]

    return pairs


def build_partA_metadata(dataset, indices: List[int], split: str) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Returns:
        caption_records: one caption per image
        vqa_records: five VQA pairs per image
    """
    caption_records = []
    vqa_records = []

    for local_id, dataset_idx in enumerate(indices):
        _, label = dataset[dataset_idx]
        cls_name = CIFAR10_LABEL_TO_NAME[label]

        caption = make_caption(label, local_id)

        caption_records.append({
            "split": split,
            "local_id": local_id,
            "dataset_idx": dataset_idx,
            "label": label,
            "class_name": cls_name,
            "caption": caption,
        })

        pairs = make_vqa_pairs_for_image(label, local_id)
        for pair_id, pair in enumerate(pairs):
            vqa_records.append({
                "split": split,
                "local_id": local_id,
                "dataset_idx": dataset_idx,
                "label": label,
                "class_name": cls_name,
                "pair_id": pair_id,
                **pair,
            })

    return caption_records, vqa_records


train_caption_records_A, train_vqa_records_A = build_partA_metadata(
    cifar_train_full,
    train_indices_A,
    split="train",
)

test_caption_records_A, val_vqa_records_A = build_partA_metadata(
    cifar_test_full,
    test_indices_A,
    split="val",
)

print("Train caption records:", len(train_caption_records_A))
print("Train VQA records:", len(train_vqa_records_A))
print("Val caption records:", len(test_caption_records_A))
print("Val VQA records:", len(val_vqa_records_A))

assert len(train_caption_records_A) == 10_000
assert len(train_vqa_records_A) == 50_000
assert len(test_caption_records_A) == 2_000
assert len(val_vqa_records_A) == 10_000

print("\nSample caption records:")
for r in train_caption_records_A[:3]:
    print(r)

print("\nSample VQA records:")
for r in train_vqa_records_A[:10]:
    print(r)

# Check yes/no balance for the class_presence template
presence_records = [r for r in train_vqa_records_A if r["template_id"] == "class_presence"]
presence_counts = {}
for r in presence_records:
    presence_counts[r["answer"]] = presence_counts.get(r["answer"], 0) + 1

print("\nClass-presence yes/no counts:", presence_counts)

print("\nPart A caption/VQA metadata checks passed.")

Load CLIP processor and frozen CLIP vision encoder

In [ ]:
reset_peak_memory()

print("Loading CLIP image processor:", partA_cfg.clip_name)
clip_processor = CLIPImageProcessor.from_pretrained(partA_cfg.clip_name)

print("CLIP processor image mean:", clip_processor.image_mean)
print("CLIP processor image std:", clip_processor.image_std)

# ImageNet defaults for comparison
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

print("\nImageNet mean:", imagenet_mean)
print("ImageNet std:", imagenet_std)

assert clip_processor.image_mean != imagenet_mean, "CLIP mean unexpectedly equals ImageNet mean."
assert clip_processor.image_std != imagenet_std, "CLIP std unexpectedly equals ImageNet std."

print("\nCLIP normalization differs from ImageNet defaults ✅")

print("\nLoading CLIP vision model:", partA_cfg.clip_name)
clip_vision = CLIPVisionModel.from_pretrained(partA_cfg.clip_name)

clip_vision.to(DEVICE)
clip_vision.eval()
set_requires_grad(clip_vision, False)

print_parameter_summary(clip_vision, "Frozen CLIP vision encoder")

# Verify all CLIP params frozen
num_trainable_clip = sum(p.numel() for p in clip_vision.parameters() if p.requires_grad)
assert num_trainable_clip == 0, "CLIP still has trainable parameters."

print("CLIP frozen check passed.")
print("GPU memory GB:", get_gpu_memory_gb())

Validate CLIP preprocessing and patch-token extraction

In [ ]:
@torch.no_grad()
def extract_clip_patch_tokens_from_pil_batch(
    images: List[Image.Image],
    processor,
    clip_model,
    device: torch.device,
    output_dtype: torch.dtype = torch.float16,
) -> torch.Tensor:
    """
    Returns CLIP patch tokens after discarding CLS.

    Expected:
        CLIP last_hidden_state: [B, 50, 768]
        patch_tokens:          [B, 49, 768]
    """
    inputs = processor(
        images=images,
        return_tensors="pt",
    )

    pixel_values = inputs["pixel_values"].to(device)

    outputs = clip_model(pixel_values=pixel_values)
    last_hidden = outputs.last_hidden_state

    # CLIP ViT-B/32 gives 1 CLS + 7x7 patches = 50 tokens
    assert last_hidden.ndim == 3
    assert last_hidden.shape[1] == 50, f"Expected 50 CLIP tokens, got {last_hidden.shape[1]}"
    assert last_hidden.shape[2] == partA_cfg.clip_hidden_size

    patch_tokens = last_hidden[:, 1:, :]

    assert patch_tokens.shape[1] == partA_cfg.clip_patch_tokens
    assert patch_tokens.shape[2] == partA_cfg.clip_hidden_size

    return patch_tokens.detach().to(output_dtype).cpu()


# Test on 8 images
sample_images = [cifar_train_full[idx][0] for idx in train_indices_A[:8]]
sample_labels = [cifar_train_full[idx][1] for idx in train_indices_A[:8]]

sample_patch_tokens = extract_clip_patch_tokens_from_pil_batch(
    images=sample_images,
    processor=clip_processor,
    clip_model=clip_vision,
    device=DEVICE,
    output_dtype=partA_cfg.clip_feature_dtype,
)

print("Sample labels:", [CIFAR10_LABEL_TO_NAME[l] for l in sample_labels])
print("Sample CLIP patch tokens shape:", sample_patch_tokens.shape)
print("Sample CLIP patch tokens dtype:", sample_patch_tokens.dtype)
print("Patch token mean:", sample_patch_tokens.float().mean().item())
print("Patch token std:", sample_patch_tokens.float().std().item())

assert sample_patch_tokens.shape == (
    8,
    partA_cfg.clip_patch_tokens,
    partA_cfg.clip_hidden_size,
)

print("\nCLIP feature extraction validation passed.")

Cache CLIP patch features for Part A

In [ ]:

def cache_clip_features_for_indices(
    dataset,
    indices: List[int],
    processor,
    clip_model,
    device: torch.device,
    save_path: str,
    batch_size: int = 128,
    output_dtype: torch.dtype = torch.float16,
):
    """
    Extracts and saves CLIP patch tokens for a list of CIFAR indices.

    Saved tensor shape:
        [N, 49, 768]
    """
    if os.path.exists(save_path):
        print(f"Cache already exists: {save_path}")
        features = torch.load(save_path, map_location="cpu")
        print("Loaded cached features shape:", features.shape)
        print("Loaded cached features dtype:", features.dtype)
        return features

    all_features = []

    clip_model.eval()
    reset_peak_memory()

    for start in tqdm(range(0, len(indices), batch_size), desc=f"Caching CLIP features -> {os.path.basename(save_path)}"):
        batch_indices = indices[start:start + batch_size]
        batch_images = [dataset[idx][0] for idx in batch_indices]

        batch_features = extract_clip_patch_tokens_from_pil_batch(
            images=batch_images,
            processor=processor,
            clip_model=clip_model,
            device=device,
            output_dtype=output_dtype,
        )

        all_features.append(batch_features)

    features = torch.cat(all_features, dim=0).contiguous()

    assert features.shape == (
        len(indices),
        partA_cfg.clip_patch_tokens,
        partA_cfg.clip_hidden_size,
    )

    torch.save(features, save_path)

    print("\nSaved cache:", save_path)
    print("Feature shape:", features.shape)
    print("Feature dtype:", features.dtype)
    print("Feature size GB:", round(features.numel() * features.element_size() / 1024**3, 3))
    print("GPU memory GB:", get_gpu_memory_gb())

    return features


train_clip_features_path_A = os.path.join(CACHE_DIR, "partA_train_clip_patch_tokens_fp16.pt")
val_clip_features_path_A = os.path.join(CACHE_DIR, "partA_val_clip_patch_tokens_fp16.pt")

train_clip_features_A = cache_clip_features_for_indices(
    dataset=cifar_train_full,
    indices=train_indices_A,
    processor=clip_processor,
    clip_model=clip_vision,
    device=DEVICE,
    save_path=train_clip_features_path_A,
    batch_size=128,
    output_dtype=partA_cfg.clip_feature_dtype,
)

val_clip_features_A = cache_clip_features_for_indices(
    dataset=cifar_test_full,
    indices=test_indices_A,
    processor=clip_processor,
    clip_model=clip_vision,
    device=DEVICE,
    save_path=val_clip_features_path_A,
    batch_size=128,
    output_dtype=partA_cfg.clip_feature_dtype,
)

print("\nTrain CLIP features:", train_clip_features_A.shape, train_clip_features_A.dtype)
print("Val CLIP features:", val_clip_features_A.shape, val_clip_features_A.dtype)

print("\nPart A CLIP feature caching completed.")

Part A caption dataset and dataloader

In [ ]:

class PartACaptionDataset(Dataset):
    """
    Dataset for Phase 1 connector initialization.

    Each item:
        clip_features: [49, 768]
        caption: text string
        class_name: CIFAR-10 class
    """
    def __init__(
        self,
        records: List[Dict[str, Any]],
        clip_features: torch.Tensor,
    ):
        assert len(records) == clip_features.shape[0]
        self.records = records
        self.clip_features = clip_features

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        return {
            "clip_features": self.clip_features[idx],
            "caption": rec["caption"],
            "class_name": rec["class_name"],
            "local_id": rec["local_id"],
            "label": rec["label"],
        }


def tokenize_text_suffix(
    tokenizer,
    texts: List[str],
    max_length: int = 64,
    add_eos: bool = True,
):
    """
    Tokenize short suffix text that appears after visual tokens.
    We use right padding here so caption content starts immediately after the visual prefix.
    """
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    if add_eos:
        texts = [t.strip() + tokenizer.eos_token for t in texts]

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        add_special_tokens=False,
    )

    tokenizer.padding_side = old_padding_side
    return enc


def collate_partA_caption_batch(batch: List[Dict[str, Any]], tokenizer, max_caption_length: int = 64):
    clip_features = torch.stack([item["clip_features"] for item in batch], dim=0)

    captions = [item["caption"] for item in batch]
    class_names = [item["class_name"] for item in batch]
    local_ids = [item["local_id"] for item in batch]

    caption_enc = tokenize_text_suffix(
        tokenizer=tokenizer,
        texts=captions,
        max_length=max_caption_length,
        add_eos=True,
    )

    return {
        "clip_features": clip_features,
        "caption_input_ids": caption_enc["input_ids"],
        "caption_attention_mask": caption_enc["attention_mask"],
        "captions": captions,
        "class_names": class_names,
        "local_ids": local_ids,
    }


phase1_caption_dataset_A = PartACaptionDataset(
    records=train_caption_records_A,
    clip_features=train_clip_features_A,
)

phase1_caption_loader_A = DataLoader(
    phase1_caption_dataset_A,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_partA_caption_batch(
        batch=batch,
        tokenizer=tokenizer,
        max_caption_length=64,
    ),
)

# Sanity check one batch
batch = next(iter(phase1_caption_loader_A))

print("Batch keys:", batch.keys())
print("clip_features:", batch["clip_features"].shape, batch["clip_features"].dtype)
print("caption_input_ids:", batch["caption_input_ids"].shape, batch["caption_input_ids"].dtype)
print("caption_attention_mask:", batch["caption_attention_mask"].shape)
print("Sample caption:", batch["captions"][0])
print("Decoded caption ids:")
print(safe_decode(tokenizer, batch["caption_input_ids"][0]))

assert batch["clip_features"].shape[1:] == (49, 768)
assert batch["caption_input_ids"].ndim == 2

print("\nPart A caption dataloader sanity check passed.")

MLP connector for Part A

In [ ]:
class MLPConnector(nn.Module):
    """
    MLP projector:
        V = Linear(768, 960) -> GELU -> Linear(960, 960)

    Input:
        clip patch tokens Z: [B, 49, 768]

    Output:
        visual tokens V: [B, 49, 960]
    """
    def __init__(self, in_dim: int = 768, hidden_dim: int = 960, out_dim: int = 960):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, a=math.sqrt(5))
                if module.bias is not None:
                    fan_in, _ = nn.init._calculate_fan_in_and_fan_out(module.weight)
                    bound = 1 / math.sqrt(fan_in)
                    nn.init.uniform_(module.bias, -bound, bound)

    def forward(self, clip_patch_tokens: torch.Tensor) -> torch.Tensor:
        return self.net(clip_patch_tokens)


connector_A = MLPConnector(
    in_dim=partA_cfg.clip_hidden_size,
    hidden_dim=partA_cfg.lm_hidden_size,
    out_dim=partA_cfg.lm_hidden_size,
).to(DEVICE)

print_parameter_summary(connector_A, "Part A MLP Connector")

# Manual expected count:
# Linear 768->960: 768*960 + 960
# Linear 960->960: 960*960 + 960
expected_connector_params = (768 * 960 + 960) + (960 * 960 + 960)
actual_connector_params = count_parameters(connector_A)["total"]

print("Expected connector params:", expected_connector_params)
print("Actual connector params:", actual_connector_params)

assert actual_connector_params == expected_connector_params
print("Connector parameter count check passed.")

Freeze LM and build Phase 1 forward pass

In [ ]:

# Freeze full LM for Phase 1
set_requires_grad(lm, False)
lm.eval()

# Connector should be trainable
set_requires_grad(connector_A, True)
connector_A.train()

print_parameter_summary(lm, "SmolLM2 frozen for Phase 1")
print_parameter_summary(connector_A, "Connector trainable for Phase 1")

def get_lm_input_embeddings(model):
    return model.get_input_embeddings()


def build_phase1_inputs_and_labels(
    lm_model,
    connector,
    clip_features: torch.Tensor,
    caption_input_ids: torch.Tensor,
    caption_attention_mask: torch.Tensor,
    device: torch.device,
):
    """
    Builds:
        inputs_embeds = [BOS emb, visual connector embs, caption token embs]
        attention_mask = 1 for BOS/visuals + caption mask
        labels = -100 for BOS/visuals/pad, caption token IDs otherwise

    Shape:
        inputs_embeds: [B, 1 + 49 + T, 960]
        labels:        [B, 1 + 49 + T]
    """
    B = clip_features.shape[0]

    clip_features = clip_features.to(device, non_blocking=True)
    caption_input_ids = caption_input_ids.to(device, non_blocking=True)
    caption_attention_mask = caption_attention_mask.to(device, non_blocking=True)

    # Connector maps CLIP patch tokens into LM hidden space
    visual_embeds = connector(clip_features)

    # Match LM dtype
    visual_embeds = visual_embeds.to(dtype=next(lm_model.parameters()).dtype)

    input_embeddings = get_lm_input_embeddings(lm_model)

    # BOS token embedding
    bos_id = tokenizer.bos_token_id
    if bos_id is None:
        bos_id = tokenizer.eos_token_id

    bos_ids = torch.full(
        (B, 1),
        fill_value=bos_id,
        dtype=torch.long,
        device=device,
    )
    bos_embeds = input_embeddings(bos_ids)

    # Caption token embeddings
    caption_embeds = input_embeddings(caption_input_ids)

    # Concatenate as required by the manual
    inputs_embeds = torch.cat(
        [bos_embeds, visual_embeds, caption_embeds],
        dim=1,
    )

    # Attention mask
    prefix_len = 1 + visual_embeds.shape[1]
    prefix_attention = torch.ones(
        (B, prefix_len),
        dtype=caption_attention_mask.dtype,
        device=device,
    )

    attention_mask = torch.cat(
        [prefix_attention, caption_attention_mask],
        dim=1,
    )

    # Labels
    prefix_labels = torch.full(
        (B, prefix_len),
        fill_value=-100,
        dtype=torch.long,
        device=device,
    )

    caption_labels = caption_input_ids.clone()
    caption_labels[caption_attention_mask == 0] = -100

    labels = torch.cat(
        [prefix_labels, caption_labels],
        dim=1,
    )

    assert inputs_embeds.shape[:2] == labels.shape
    assert attention_mask.shape == labels.shape
    assert inputs_embeds.shape[-1] == cfg.expected_lm_hidden_size

    return inputs_embeds, attention_mask, labels


# Forward sanity check
batch = next(iter(phase1_caption_loader_A))

with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
    inputs_embeds, attention_mask, labels = build_phase1_inputs_and_labels(
        lm_model=lm,
        connector=connector_A,
        clip_features=batch["clip_features"],
        caption_input_ids=batch["caption_input_ids"],
        caption_attention_mask=batch["caption_attention_mask"],
        device=DEVICE,
    )

    outputs = lm(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        labels=labels,
        use_cache=False,
    )

print("inputs_embeds:", inputs_embeds.shape, inputs_embeds.dtype)
print("attention_mask:", attention_mask.shape)
print("labels:", labels.shape)
print("Phase 1 initial loss:", outputs.loss.item())

# Check that only caption positions have labels
num_supervised = (labels != -100).sum().item()
num_total = labels.numel()
print("Supervised tokens:", num_supervised)
print("Total positions:", num_total)
print("Supervised fraction:", round(num_supervised / num_total, 4))

assert outputs.loss.item() > 0
assert num_supervised > 0
print("\nPhase 1 forward/loss sanity check passed.")

Norm-ratio utility

In [ ]:
@torch.no_grad()
def compute_connector_rnorm(
    connector,
    lm_model,
    data_loader,
    tokenizer,
    device: torch.device,
    num_batches: int = 10,
) -> Dict[str, float]:
    """
    rnorm = E||visual token|| / E||text token||

    Uses caption token embeddings as the text-token reference.
    """
    connector.eval()
    lm_model.eval()

    visual_norms = []
    text_norms = []

    input_embeddings = get_lm_input_embeddings(lm_model)

    for i, batch in enumerate(data_loader):
        if i >= num_batches:
            break

        clip_features = batch["clip_features"].to(device, non_blocking=True)
        caption_input_ids = batch["caption_input_ids"].to(device, non_blocking=True)
        caption_attention_mask = batch["caption_attention_mask"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            visual_embeds = connector(clip_features)
            text_embeds = input_embeddings(caption_input_ids)

        # valid caption tokens only
        valid_text = caption_attention_mask.bool()

        visual_norms.append(visual_embeds.float().norm(dim=-1).reshape(-1).cpu())
        text_norms.append(text_embeds.float().norm(dim=-1)[valid_text].reshape(-1).cpu())

    visual_norms = torch.cat(visual_norms)
    text_norms = torch.cat(text_norms)

    visual_mean = visual_norms.mean().item()
    text_mean = text_norms.mean().item()
    rnorm = visual_mean / max(text_mean, 1e-8)

    return {
        "visual_norm_mean": visual_mean,
        "text_norm_mean": text_mean,
        "rnorm": rnorm,
    }


def rescale_connector_output_layer(connector: MLPConnector, scale: float):
    """
    Rescale final linear layer so visual output norms move closer to text norms.
    """
    final_linear = connector.net[-1]
    assert isinstance(final_linear, nn.Linear)

    with torch.no_grad():
        final_linear.weight.mul_(scale)
        if final_linear.bias is not None:
            final_linear.bias.mul_(scale)


rnorm_before = compute_connector_rnorm(
    connector=connector_A,
    lm_model=lm,
    data_loader=phase1_caption_loader_A,
    tokenizer=tokenizer,
    device=DEVICE,
    num_batches=5,
)

print("Initial connector rnorm stats:")
print(rnorm_before)

if rnorm_before["rnorm"] < 0.3 or rnorm_before["rnorm"] > 3.0:
    scale = 1.0 / rnorm_before["rnorm"]
    print(f"rnorm outside [0.3, 3.0]. Rescaling final connector layer by {scale:.4f}")
    rescale_connector_output_layer(connector_A, scale)

    rnorm_after = compute_connector_rnorm(
        connector=connector_A,
        lm_model=lm,
        data_loader=phase1_caption_loader_A,
        tokenizer=tokenizer,
        device=DEVICE,
        num_batches=5,
    )
    print("After rescale rnorm stats:")
    print(rnorm_after)
else:
    print("Initial rnorm is acceptable. No rescale needed.")

Phase 1 training loop

In [ ]:
# ============================================================
# Cell 19: Phase 1 training loop
# ============================================================

def train_phase1_connector(
    lm_model,
    connector,
    train_loader,
    optimizer,
    device: torch.device,
    num_epochs: int = 1,
    log_every: int = 50,
):
    """
    Phase 1:
        Train only connector on caption pairs.
        CLIP features are cached.
        LM is frozen.
    """
    lm_model.eval()
    connector.train()

    scaler = torch.amp.GradScaler("cuda")

    history = {
        "step": [],
        "loss": [],
        "lr": [],
    }

    global_step = 0
    start_time = time.time()
    reset_peak_memory()

    for epoch in range(num_epochs):
        pbar = tqdm(train_loader, desc=f"Phase 1 Epoch {epoch+1}/{num_epochs}")

        running_loss = 0.0

        for step, batch in enumerate(pbar):
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                inputs_embeds, attention_mask, labels = build_phase1_inputs_and_labels(
                    lm_model=lm_model,
                    connector=connector,
                    clip_features=batch["clip_features"],
                    caption_input_ids=batch["caption_input_ids"],
                    caption_attention_mask=batch["caption_attention_mask"],
                    device=device,
                )

                outputs = lm_model(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    labels=labels,
                    use_cache=False,
                )

                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(connector.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            loss_value = loss.detach().float().item()
            running_loss += loss_value
            global_step += 1

            pbar.set_postfix({
                "loss": f"{loss_value:.4f}",
                "avg": f"{running_loss / (step + 1):.4f}",
            })

            if global_step % log_every == 0:
                lr = optimizer.param_groups[0]["lr"]
                history["step"].append(global_step)
                history["loss"].append(running_loss / (step + 1))
                history["lr"].append(lr)

                print(
                    f"step={global_step:04d} | "
                    f"avg_loss={running_loss / (step + 1):.4f} | "
                    f"lr={lr:.2e} | "
                    f"mem={get_gpu_memory_gb()}"
                )

    elapsed = time.time() - start_time

    print("\nPhase 1 training complete.")
    print("Elapsed minutes:", round(elapsed / 60, 2))
    print("Peak GPU memory GB:", get_gpu_memory_gb()["max_allocated"])

    return history


phase1_optimizer_A = torch.optim.AdamW(
    connector_A.parameters(),
    lr=3e-4,
    weight_decay=0.01,
)

phase1_history_A = train_phase1_connector(
    lm_model=lm,
    connector=connector_A,
    train_loader=phase1_caption_loader_A,
    optimizer=phase1_optimizer_A,
    device=DEVICE,
    num_epochs=1,
    log_every=50,
)

Plot Phase 1 loss and compute final rnorm


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(phase1_history_A["step"], phase1_history_A["loss"], marker="o")
plt.xlabel("Training step")
plt.ylabel("Average loss")
plt.title("Part A Phase 1 Connector Caption Loss")
plt.grid(True)
plt.show()

rnorm_phase1 = compute_connector_rnorm(
    connector=connector_A,
    lm_model=lm,
    data_loader=phase1_caption_loader_A,
    tokenizer=tokenizer,
    device=DEVICE,
    num_batches=10,
)

print("Phase 1 rnorm stats:")
print(rnorm_phase1)

if rnorm_phase1["rnorm"] < 0.3 or rnorm_phase1["rnorm"] > 3.0:
    print("WARNING: rnorm outside broad safety range [0.3, 3.0]. Consider rescaling.")
elif rnorm_phase1["rnorm"] < 0.5 or rnorm_phase1["rnorm"] > 2.0:
    print("NOTE: rnorm outside recommended range [0.5, 2.0], but inside broad safety range.")
else:
    print("rnorm is inside recommended range [0.5, 2.0]. ✅")

Greedy caption generation from visual tokens

In [ ]:
@torch.no_grad()
def generate_caption_from_clip_features(
    lm_model,
    connector,
    clip_features: torch.Tensor,
    tokenizer,
    device: torch.device,
    max_new_tokens: int = 24,
):
    """
    Greedy generation using only:
        [BOS emb, visual embeddings]

    Then autoregressively appends generated token embeddings.
    """
    lm_model.eval()
    connector.eval()

    clip_features = clip_features.unsqueeze(0).to(device)

    input_embeddings = get_lm_input_embeddings(lm_model)

    bos_id = tokenizer.bos_token_id
    if bos_id is None:
        bos_id = tokenizer.eos_token_id

    bos_ids = torch.tensor([[bos_id]], dtype=torch.long, device=device)
    bos_embeds = input_embeddings(bos_ids)

    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        visual_embeds = connector(clip_features)
        visual_embeds = visual_embeds.to(dtype=next(lm_model.parameters()).dtype)

    inputs_embeds = torch.cat([bos_embeds, visual_embeds], dim=1)
    attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)

    generated_ids = []

    for _ in range(max_new_tokens):
        outputs = lm_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            use_cache=False,
        )

        next_logits = outputs.logits[:, -1, :]
        next_id = torch.argmax(next_logits, dim=-1)

        if next_id.item() == tokenizer.eos_token_id:
            break

        generated_ids.append(next_id.item())

        next_embed = input_embeddings(next_id[:, None])
        inputs_embeds = torch.cat([inputs_embeds, next_embed], dim=1)

        next_attn = torch.ones((1, 1), dtype=torch.long, device=device)
        attention_mask = torch.cat([attention_mask, next_attn], dim=1)

    decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return decoded.strip(), generated_ids


# Generate 5 captions from held-out val images
print("Held-out Phase 1 greedy generations:")
print("=" * 80)

for i in range(5):
    rec = test_caption_records_A[i]
    features = val_clip_features_A[i]

    pred_caption, pred_ids = generate_caption_from_clip_features(
        lm_model=lm,
        connector=connector_A,
        clip_features=features,
        tokenizer=tokenizer,
        device=DEVICE,
        max_new_tokens=24,
    )

    print(f"\nExample {i+1}")
    print("Class:", rec["class_name"])
    print("Reference caption:", rec["caption"])
    print("Generated caption:", repr(pred_caption))

Save Phase 1 connector checkpoint

In [ ]:
phase1_ckpt_path_A = os.path.join(WEIGHTS_DIR, "connector_phaseA1.pt")

phase1_payload_A = {
    "connector_state_dict": connector_A.state_dict(),
    "partA_config": partA_cfg.__dict__,
    "global_config": cfg.__dict__,
    "rnorm_phase1": rnorm_phase1,
    "phase1_history": phase1_history_A,
}

torch.save(phase1_payload_A, phase1_ckpt_path_A)

print("Saved Phase 1 connector checkpoint:")
print(phase1_ckpt_path_A)

# Verify load works
loaded_payload = torch.load(phase1_ckpt_path_A, map_location="cpu")
print("Checkpoint keys:", loaded_payload.keys())
print("Saved rnorm:", loaded_payload["rnorm_phase1"])

Rescale Phase 1 connector output norm

In [ ]:

current_rnorm = rnorm_phase1["rnorm"]

# Target a slightly conservative value around 1.0.
# This means average visual-token norm roughly matches text-token norm.
target_rnorm = 1.0

scale = target_rnorm / current_rnorm

print("Current rnorm:", current_rnorm)
print("Target rnorm:", target_rnorm)
print("Scaling final connector layer by:", scale)

rescale_connector_output_layer(connector_A, scale)

rnorm_phase1_rescaled = compute_connector_rnorm(
    connector=connector_A,
    lm_model=lm,
    data_loader=phase1_caption_loader_A,
    tokenizer=tokenizer,
    device=DEVICE,
    num_batches=10,
)

print("\nRescaled Phase 1 rnorm stats:")
print(rnorm_phase1_rescaled)

if 0.5 <= rnorm_phase1_rescaled["rnorm"] <= 2.0:
    print("rnorm is now inside recommended range [0.5, 2.0]. ✅")
elif 0.3 <= rnorm_phase1_rescaled["rnorm"] <= 3.0:
    print("rnorm is inside broad safety range [0.3, 3.0]. ✅")
else:
    print("WARNING: rnorm still outside broad safety range.")

Re-test held-out caption generation after rescaling

In [ ]:

print("Held-out Phase 1 greedy generations AFTER rnorm rescale:")
print("=" * 80)

rescaled_generations = []

for i in range(5):
    rec = test_caption_records_A[i]
    features = val_clip_features_A[i]

    pred_caption, pred_ids = generate_caption_from_clip_features(
        lm_model=lm,
        connector=connector_A,
        clip_features=features,
        tokenizer=tokenizer,
        device=DEVICE,
        max_new_tokens=24,
    )

    rescaled_generations.append({
        "class": rec["class_name"],
        "reference": rec["caption"],
        "generated": pred_caption,
    })

    print(f"\nExample {i+1}")
    print("Class:", rec["class_name"])
    print("Reference caption:", rec["caption"])
    print("Generated caption:", repr(pred_caption))

Save corrected Phase 1 checkpoint

In [ ]:
phase1_corrected_ckpt_path_A = os.path.join(WEIGHTS_DIR, "connector_phaseA1_rescaled.pt")

phase1_corrected_payload_A = {
    "connector_state_dict": connector_A.state_dict(),
    "partA_config": partA_cfg.__dict__,
    "global_config": cfg.__dict__,
    "rnorm_phase1_original": rnorm_phase1,
    "rnorm_phase1_rescaled": rnorm_phase1_rescaled,
    "phase1_history": phase1_history_A,
    "rescale_factor": scale,
    "rescaled_generations": rescaled_generations,
}

torch.save(phase1_corrected_payload_A, phase1_corrected_ckpt_path_A)

print("Saved corrected Phase 1 connector checkpoint:")
print(phase1_corrected_ckpt_path_A)

loaded_payload = torch.load(phase1_corrected_ckpt_path_A, map_location="cpu")
print("Checkpoint keys:", loaded_payload.keys())
print("Original rnorm:", loaded_payload["rnorm_phase1_original"])
print("Rescaled rnorm:", loaded_payload["rnorm_phase1_rescaled"])
print("Rescale factor:", loaded_payload["rescale_factor"])

Restore original Phase 1 connector and softly rescale to rnorm 2.0

In [ ]:

# Reload original high-rnorm Phase 1 checkpoint, not the target-1.0 checkpoint
original_phase1_ckpt = torch.load(
    phase1_ckpt_path_A,
    map_location=DEVICE,
)

connector_A.load_state_dict(original_phase1_ckpt["connector_state_dict"])
connector_A.to(DEVICE)

# Recompute to verify we are back to original high-rnorm connector
rnorm_original_reloaded = compute_connector_rnorm(
    connector=connector_A,
    lm_model=lm,
    data_loader=phase1_caption_loader_A,
    tokenizer=tokenizer,
    device=DEVICE,
    num_batches=10,
)

print("Reloaded original Phase 1 rnorm:")
print(rnorm_original_reloaded)

# Soft target: preserve visual influence but bring norm into recommended upper edge
target_rnorm_soft = 2.0
scale_soft = target_rnorm_soft / rnorm_original_reloaded["rnorm"]

print("\nSoft target rnorm:", target_rnorm_soft)
print("Soft rescale factor:", scale_soft)

rescale_connector_output_layer(connector_A, scale_soft)

rnorm_phase1_soft_rescaled = compute_connector_rnorm(
    connector=connector_A,
    lm_model=lm,
    data_loader=phase1_caption_loader_A,
    tokenizer=tokenizer,
    device=DEVICE,
    num_batches=10,
)

print("\nSoft-rescaled Phase 1 rnorm stats:")
print(rnorm_phase1_soft_rescaled)

if 0.5 <= rnorm_phase1_soft_rescaled["rnorm"] <= 2.1:
    print("Soft-rescaled rnorm is acceptable for Phase 2. ✅")
else:
    print("WARNING: Soft-rescaled rnorm is still outside target range.")

Test generation with soft-rescaled connector

In [ ]:

print("Held-out Phase 1 greedy generations AFTER soft rnorm rescale:")
print("=" * 80)

soft_rescaled_generations = []

for i in range(5):
    rec = test_caption_records_A[i]
    features = val_clip_features_A[i]

    pred_caption, pred_ids = generate_caption_from_clip_features(
        lm_model=lm,
        connector=connector_A,
        clip_features=features,
        tokenizer=tokenizer,
        device=DEVICE,
        max_new_tokens=24,
    )

    soft_rescaled_generations.append({
        "class": rec["class_name"],
        "reference": rec["caption"],
        "generated": pred_caption,
    })

    print(f"\nExample {i+1}")
    print("Class:", rec["class_name"])
    print("Reference caption:", rec["caption"])
    print("Generated caption:", repr(pred_caption))

Save soft-rescaled connector for Phase 2

In [ ]:

phase1_soft_ckpt_path_A = os.path.join(WEIGHTS_DIR, "connector_phaseA1_soft_rescaled.pt")

phase1_soft_payload_A = {
    "connector_state_dict": connector_A.state_dict(),
    "partA_config": partA_cfg.__dict__,
    "global_config": cfg.__dict__,
    "rnorm_phase1_original": rnorm_original_reloaded,
    "rnorm_phase1_soft_rescaled": rnorm_phase1_soft_rescaled,
    "phase1_history": phase1_history_A,
    "soft_target_rnorm": target_rnorm_soft,
    "soft_rescale_factor": scale_soft,
    "soft_rescaled_generations": soft_rescaled_generations,
}

torch.save(phase1_soft_payload_A, phase1_soft_ckpt_path_A)

print("Saved soft-rescaled Phase 1 connector checkpoint:")
print(phase1_soft_ckpt_path_A)

loaded_payload = torch.load(phase1_soft_ckpt_path_A, map_location="cpu")
print("Checkpoint keys:", loaded_payload.keys())
print("Original rnorm:", loaded_payload["rnorm_phase1_original"])
print("Soft-rescaled rnorm:", loaded_payload["rnorm_phase1_soft_rescaled"])
print("Soft target rnorm:", loaded_payload["soft_target_rnorm"])
print("Soft rescale factor:", loaded_payload["soft_rescale_factor"])

Part A VQA dataset and collator

In [ ]:

class PartAVQADataset(Dataset):
    """
    Part A VQA dataset.

    Each item:
        clip_features: [49, 768]
        question: str
        answer: str
        template_id: str
        class_name: str
    """
    def __init__(
        self,
        records: List[Dict[str, Any]],
        clip_features: torch.Tensor,
    ):
        self.records = records
        self.clip_features = clip_features

        # records are 5 per image, so local_id maps to clip_features row
        assert max(r["local_id"] for r in records) < clip_features.shape[0]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        local_id = rec["local_id"]

        return {
            "clip_features": self.clip_features[local_id],
            "question": rec["question"],
            "answer": rec["answer"],
            "template_id": rec["template_id"],
            "class_name": rec["class_name"],
            "label": rec["label"],
            "local_id": local_id,
        }


def collate_partA_vqa_batch(
    batch: List[Dict[str, Any]],
    tokenizer,
    max_question_length: int = 48,
    max_answer_length: int = 16,
):
    clip_features = torch.stack([item["clip_features"] for item in batch], dim=0)

    questions = [item["question"] for item in batch]
    answers = [item["answer"] for item in batch]

    # Add a leading space before answer so tokenization is more natural after question text.
    # Add EOS so answer generation has a learned stop.
    answer_texts = [" " + a.strip() + tokenizer.eos_token for a in answers]

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    q_enc = tokenizer(
        questions,
        padding=True,
        truncation=True,
        max_length=max_question_length,
        return_tensors="pt",
        add_special_tokens=False,
    )

    a_enc = tokenizer(
        answer_texts,
        padding=True,
        truncation=True,
        max_length=max_answer_length,
        return_tensors="pt",
        add_special_tokens=False,
    )

    tokenizer.padding_side = old_padding_side

    return {
        "clip_features": clip_features,
        "question_input_ids": q_enc["input_ids"],
        "question_attention_mask": q_enc["attention_mask"],
        "answer_input_ids": a_enc["input_ids"],
        "answer_attention_mask": a_enc["attention_mask"],
        "questions": questions,
        "answers": answers,
        "template_ids": [item["template_id"] for item in batch],
        "class_names": [item["class_name"] for item in batch],
        "labels_class": [item["label"] for item in batch],
        "local_ids": [item["local_id"] for item in batch],
    }


train_vqa_dataset_A = PartAVQADataset(
    records=train_vqa_records_A,
    clip_features=train_clip_features_A,
)

val_vqa_dataset_A = PartAVQADataset(
    records=val_vqa_records_A,
    clip_features=val_clip_features_A,
)

train_vqa_loader_A = DataLoader(
    train_vqa_dataset_A,
    batch_size=16,          # A100 can handle larger, but 16 + accum is safe
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_partA_vqa_batch(batch, tokenizer),
)

val_vqa_loader_A_small = DataLoader(
    torch.utils.data.Subset(val_vqa_dataset_A, list(range(200))),
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_partA_vqa_batch(batch, tokenizer),
)

batch = next(iter(train_vqa_loader_A))

print("VQA batch keys:", batch.keys())
print("clip_features:", batch["clip_features"].shape, batch["clip_features"].dtype)
print("question_input_ids:", batch["question_input_ids"].shape)
print("answer_input_ids:", batch["answer_input_ids"].shape)
print("Sample Q:", batch["questions"][0])
print("Sample A:", batch["answers"][0])
print("Decoded Q:", safe_decode(tokenizer, batch["question_input_ids"][0]))
print("Decoded A:", safe_decode(tokenizer, batch["answer_input_ids"][0]))

assert batch["clip_features"].shape[1:] == (49, 768)
print("\nPart A VQA dataloader sanity check passed.")

Alpaca replay dataset and collator

In [ ]:
class AlpacaReplayDataset(Dataset):
    """
    Pure-text replay dataset.
    Important: this dataset must never include visual tokens/embeddings.
    """
    def __init__(self, texts: List[str]):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx]}


def collate_alpaca_replay_batch(
    batch: List[Dict[str, str]],
    tokenizer,
    max_length: int = 256,
):
    texts = [item["text"] for item in batch]

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        add_special_tokens=True,
    )

    tokenizer.padding_side = old_padding_side

    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": labels,
        "texts": texts,
    }


alpaca_replay_dataset = AlpacaReplayDataset(alpaca_texts)

alpaca_replay_loader = DataLoader(
    alpaca_replay_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_alpaca_replay_batch(
        batch=batch,
        tokenizer=tokenizer,
        max_length=cfg.max_text_length,
    ),
)

replay_batch = next(iter(alpaca_replay_loader))

print("Replay input_ids:", replay_batch["input_ids"].shape)
print("Replay attention_mask:", replay_batch["attention_mask"].shape)
print("Replay labels:", replay_batch["labels"].shape)
print("Replay supervised tokens:", (replay_batch["labels"] != -100).sum().item())
print("Replay sample:")
print(safe_decode(tokenizer, replay_batch["input_ids"][0]))

# Guard: no Part B visual IDs exist yet, but this keeps the intent explicit.
assert replay_batch["input_ids"].max().item() < cfg.expected_lm_vocab_size

print("\nAlpaca replay dataloader sanity check passed.")

Build Phase 2 VQA inputs and labels

In [ ]:
def build_phase2_vqa_inputs_and_labels(
    lm_model,
    connector,
    clip_features: torch.Tensor,
    question_input_ids: torch.Tensor,
    question_attention_mask: torch.Tensor,
    answer_input_ids: torch.Tensor,
    answer_attention_mask: torch.Tensor,
    device: torch.device,
):
    """
    Required VQA sequence:
        [BOS emb, V1:49, question embs, answer embs]

    Required labels:
        -100 for BOS
        -100 for all visual tokens
        -100 for question tokens
        answer token IDs + EOS get labels
        pad positions get -100
    """
    B = clip_features.shape[0]

    clip_features = clip_features.to(device, non_blocking=True)
    question_input_ids = question_input_ids.to(device, non_blocking=True)
    question_attention_mask = question_attention_mask.to(device, non_blocking=True)
    answer_input_ids = answer_input_ids.to(device, non_blocking=True)
    answer_attention_mask = answer_attention_mask.to(device, non_blocking=True)

    input_embeddings = get_lm_input_embeddings(lm_model)

    visual_embeds = connector(clip_features)
    visual_embeds = visual_embeds.to(dtype=next(lm_model.parameters()).dtype)

    bos_id = tokenizer.bos_token_id
    if bos_id is None:
        bos_id = tokenizer.eos_token_id

    bos_ids = torch.full(
        (B, 1),
        fill_value=bos_id,
        dtype=torch.long,
        device=device,
    )

    bos_embeds = input_embeddings(bos_ids)
    question_embeds = input_embeddings(question_input_ids)
    answer_embeds = input_embeddings(answer_input_ids)

    inputs_embeds = torch.cat(
        [bos_embeds, visual_embeds, question_embeds, answer_embeds],
        dim=1,
    )

    prefix_len = 1 + visual_embeds.shape[1]

    bos_visual_attention = torch.ones(
        (B, prefix_len),
        dtype=torch.long,
        device=device,
    )

    attention_mask = torch.cat(
        [bos_visual_attention, question_attention_mask, answer_attention_mask],
        dim=1,
    )

    bos_visual_labels = torch.full(
        (B, prefix_len),
        fill_value=-100,
        dtype=torch.long,
        device=device,
    )

    question_labels = torch.full_like(question_input_ids, fill_value=-100)

    answer_labels = answer_input_ids.clone()
    answer_labels[answer_attention_mask == 0] = -100

    labels = torch.cat(
        [bos_visual_labels, question_labels, answer_labels],
        dim=1,
    )

    assert inputs_embeds.shape[:2] == labels.shape
    assert attention_mask.shape == labels.shape

    return inputs_embeds, attention_mask, labels


# Sanity forward before applying LoRA
batch = next(iter(train_vqa_loader_A))

with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        inputs_embeds, attention_mask, labels = build_phase2_vqa_inputs_and_labels(
            lm_model=lm,
            connector=connector_A,
            clip_features=batch["clip_features"],
            question_input_ids=batch["question_input_ids"],
            question_attention_mask=batch["question_attention_mask"],
            answer_input_ids=batch["answer_input_ids"],
            answer_attention_mask=batch["answer_attention_mask"],
            device=DEVICE,
        )

        outputs = lm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )

print("Phase 2 VQA inputs_embeds:", inputs_embeds.shape)
print("Phase 2 VQA labels:", labels.shape)
print("Initial VQA loss before LoRA training:", outputs.loss.item())
print("Supervised VQA tokens:", (labels != -100).sum().item())
print("Total VQA positions:", labels.numel())
print("Supervised fraction:", round((labels != -100).sum().item() / labels.numel(), 4))

assert outputs.loss.item() > 0
print("\nPhase 2 VQA input/label sanity check passed.")

Apply LoRA for Phase 2

In [ ]:
# Fix PEFT / torchao version mismatch


!pip -q install --upgrade "torchao>=0.16.0"

import importlib.metadata as importlib_metadata

try:
    print("torchao version:", importlib_metadata.version("torchao"))
except Exception as e:
    print("Could not read torchao version:", e)

try:
    print("peft version:", importlib_metadata.version("peft"))
except Exception as e:
    print("Could not read peft version:", e)



In [ ]:
# Important: ensure LM base is frozen before LoRA
set_requires_grad(lm, False)

lora_config_A = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=list(cfg.lora_targets),
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

lm_lora_A = get_peft_model(lm, lora_config_A)
lm_lora_A.to(DEVICE)
lm_lora_A.train()

# Connector remains trainable
connector_A.train()
set_requires_grad(connector_A, True)

print_parameter_summary(lm_lora_A, "SmolLM2 with LoRA for Phase 2")
print_parameter_summary(connector_A, "Connector for Phase 2")

# PEFT helper print if available
try:
    lm_lora_A.print_trainable_parameters()
except Exception as e:
    print("Could not call print_trainable_parameters:", e)

# Total trainable across LM LoRA + connector
total_trainable_phase2 = (
    sum(p.numel() for p in lm_lora_A.parameters() if p.requires_grad)
    + sum(p.numel() for p in connector_A.parameters() if p.requires_grad)
)

total_params_phase2 = (
    sum(p.numel() for p in lm_lora_A.parameters())
    + sum(p.numel() for p in connector_A.parameters())
)

print("\nTotal trainable Phase 2 params including connector:", f"{total_trainable_phase2:,}")
print("Total params including connector:", f"{total_params_phase2:,}")
print("Trainable % including connector:", round(100 * total_trainable_phase2 / total_params_phase2, 4))

Phase 2 quick evaluation helpers

In [ ]:
ANSWER_SPACE_A = sorted(list(set([r["answer"] for r in train_vqa_records_A + val_vqa_records_A])))
print("Answer space:", ANSWER_SPACE_A)

def normalize_answer(text: str) -> str:
    text = text.strip().lower()
    text = text.replace(tokenizer.eos_token or "", "")
    text = text.strip()

    # Keep only first simple phrase/word for exact-match robustness
    for sep in ["\n", ".", ",", ";", ":"]:
        if sep in text:
            text = text.split(sep)[0].strip()

    # Map common synonym produced in Phase 1
    synonym_map = {
        "car": "automobile",
        "auto": "automobile",
        "plane": "airplane",
        "aeroplane": "airplane",
    }
    return synonym_map.get(text, text)


@torch.no_grad()
def generate_vqa_answer_partA(
    lm_model,
    connector,
    clip_features: torch.Tensor,
    question: str,
    tokenizer,
    device: torch.device,
    max_new_tokens: int = 6,
):
    """
    Greedy VQA answer generation:
        [BOS, visual tokens, question] -> answer
    """
    lm_model.eval()
    connector.eval()

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    q_enc = tokenizer(
        [question],
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=48,
    )

    tokenizer.padding_side = old_padding_side

    clip_features = clip_features.unsqueeze(0).to(device)
    question_input_ids = q_enc["input_ids"].to(device)
    question_attention_mask = q_enc["attention_mask"].to(device)

    input_embeddings = get_lm_input_embeddings(lm_model)

    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
    bos_ids = torch.full((1, 1), bos_id, dtype=torch.long, device=device)
    bos_embeds = input_embeddings(bos_ids)

    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        visual_embeds = connector(clip_features)
        visual_embeds = visual_embeds.to(dtype=next(lm_model.parameters()).dtype)

    q_embeds = input_embeddings(question_input_ids)

    inputs_embeds = torch.cat([bos_embeds, visual_embeds, q_embeds], dim=1)
    attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)

    generated_ids = []

    for _ in range(max_new_tokens):
        outputs = lm_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            use_cache=False,
        )

        logits = outputs.logits[:, -1, :]
        next_id = torch.argmax(logits, dim=-1)

        if next_id.item() == tokenizer.eos_token_id:
            break

        generated_ids.append(next_id.item())

        next_embed = input_embeddings(next_id[:, None])
        inputs_embeds = torch.cat([inputs_embeds, next_embed], dim=1)

        next_attn = torch.ones((1, 1), dtype=torch.long, device=device)
        attention_mask = torch.cat([attention_mask, next_attn], dim=1)

    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return raw.strip(), normalize_answer(raw)


@torch.no_grad()
def evaluate_vqa_partA_small(
    lm_model,
    connector,
    dataset,
    max_examples: int = 200,
):
    lm_model.eval()
    connector.eval()

    n = min(max_examples, len(dataset))
    correct = 0
    rows = []

    for i in tqdm(range(n), desc=f"Quick VQA eval {n}"):
        item = dataset[i]
        raw_pred, norm_pred = generate_vqa_answer_partA(
            lm_model=lm_model,
            connector=connector,
            clip_features=item["clip_features"],
            question=item["question"],
            tokenizer=tokenizer,
            device=DEVICE,
            max_new_tokens=6,
        )

        gold = normalize_answer(item["answer"])
        is_correct = norm_pred == gold
        correct += int(is_correct)

        rows.append({
            "i": i,
            "question": item["question"],
            "gold": gold,
            "raw_pred": raw_pred,
            "norm_pred": norm_pred,
            "correct": is_correct,
            "template_id": item["template_id"],
            "class_name": item["class_name"],
        })

    acc = correct / max(n, 1)
    return acc, rows

Phase 2 training loop with replay

In [ ]:
def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch


def train_phase2_vqa_with_replay(
    lm_model,
    connector,
    vqa_loader,
    replay_loader,
    val_dataset,
    optimizer,
    scheduler,
    device: torch.device,
    lambda_replay: float = 0.2,
    num_epochs: int = 1,
    grad_accum_steps: int = 4,
    log_every: int = 25,
    eval_every: int = 300,
    max_train_steps: Optional[int] = None,
):
    """
    Phase 2:
        Lmixed = LVQA + lambda * LLM

    Uses sequential VQA and replay forward passes to keep memory stable.
    """
    lm_model.train()
    connector.train()

    scaler = torch.amp.GradScaler("cuda")
    replay_iter = infinite_loader(replay_loader)

    history = {
        "step": [],
        "vqa_loss": [],
        "replay_loss": [],
        "mixed_loss": [],
        "lr": [],
        "quick_vqa_acc": [],
        "ppl_replay": [],
        "R": [],
    }

    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    reset_peak_memory()
    start_time = time.time()

    total_batches = len(vqa_loader) * num_epochs
    if max_train_steps is not None:
        total_batches = min(total_batches, max_train_steps)

    pbar_total = tqdm(total=total_batches, desc="Phase 2 training")

    stop_training = False

    for epoch in range(num_epochs):
        for batch_idx, vqa_batch in enumerate(vqa_loader):
            if max_train_steps is not None and global_step >= max_train_steps:
                stop_training = True
                break

            replay_batch = next(replay_iter)

            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                # VQA loss
                inputs_embeds, attention_mask, labels = build_phase2_vqa_inputs_and_labels(
                    lm_model=lm_model,
                    connector=connector,
                    clip_features=vqa_batch["clip_features"],
                    question_input_ids=vqa_batch["question_input_ids"],
                    question_attention_mask=vqa_batch["question_attention_mask"],
                    answer_input_ids=vqa_batch["answer_input_ids"],
                    answer_attention_mask=vqa_batch["answer_attention_mask"],
                    device=device,
                )

                vqa_outputs = lm_model(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    labels=labels,
                    use_cache=False,
                )

                vqa_loss = vqa_outputs.loss

                # Pure text replay loss
                replay_input_ids = replay_batch["input_ids"].to(device, non_blocking=True)
                replay_attention_mask = replay_batch["attention_mask"].to(device, non_blocking=True)
                replay_labels = replay_batch["labels"].to(device, non_blocking=True)

                # Guard: replay must be pure text IDs within original vocab.
                assert replay_input_ids.max().item() < cfg.expected_lm_vocab_size

                replay_outputs = lm_model(
                    input_ids=replay_input_ids,
                    attention_mask=replay_attention_mask,
                    labels=replay_labels,
                    use_cache=False,
                )

                replay_loss = replay_outputs.loss

                mixed_loss = vqa_loss + lambda_replay * replay_loss
                loss_for_backward = mixed_loss / grad_accum_steps

            scaler.scale(loss_for_backward).backward()

            if (global_step + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)

                trainable_params = [
                    p for p in list(lm_model.parameters()) + list(connector.parameters())
                    if p.requires_grad and p.grad is not None
                ]

                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if scheduler is not None:
                    scheduler.step()

            global_step += 1
            pbar_total.update(1)

            if global_step % log_every == 0:
                lr = optimizer.param_groups[0]["lr"]

                print(
                    f"step={global_step:04d} | "
                    f"vqa={vqa_loss.item():.4f} | "
                    f"replay={replay_loss.item():.4f} | "
                    f"mixed={mixed_loss.item():.4f} | "
                    f"lr={lr:.2e} | "
                    f"mem={get_gpu_memory_gb()}"
                )

                history["step"].append(global_step)
                history["vqa_loss"].append(float(vqa_loss.detach().cpu()))
                history["replay_loss"].append(float(replay_loss.detach().cpu()))
                history["mixed_loss"].append(float(mixed_loss.detach().cpu()))
                history["lr"].append(lr)

            if global_step % eval_every == 0:
                quick_acc, _ = evaluate_vqa_partA_small(
                    lm_model=lm_model,
                    connector=connector,
                    dataset=val_dataset,
                    max_examples=200,
                )

                # Fast replay PPL on 100 examples
                ppl_fine = compute_perplexity(
                    model=lm_model,
                    tokenizer=tokenizer,
                    texts=ppl0_texts,
                    device=device,
                    batch_size=4,
                    max_length=cfg.max_text_length,
                    desc=f"Replay PPL step {global_step}",
                )

                R = ppl_fine / PPL0

                print(
                    f"\n[EVAL step={global_step}] "
                    f"quick_vqa_acc={quick_acc:.4f} | "
                    f"PPL={ppl_fine:.4f} | "
                    f"R={R:.4f}\n"
                )

                history["quick_vqa_acc"].append((global_step, quick_acc))
                history["ppl_replay"].append((global_step, ppl_fine))
                history["R"].append((global_step, R))

                lm_model.train()
                connector.train()

        if stop_training:
            break

    pbar_total.close()

    elapsed = time.time() - start_time

    print("\nPhase 2 training complete.")
    print("Elapsed minutes:", round(elapsed / 60, 2))
    print("Peak GPU memory GB:", get_gpu_memory_gb()["max_allocated"])

    return history

Prepare optimizer, scheduler, and run Phase 2 baseline

In [ ]:

# Optimizer parameter groups:
# connector and LoRA can share lr=5e-4 as recommended.
phase2_trainable_params = [
    p for p in list(lm_lora_A.parameters()) + list(connector_A.parameters())
    if p.requires_grad
]

phase2_optimizer_A = torch.optim.AdamW(
    phase2_trainable_params,
    lr=5e-4,
    weight_decay=0.01,
)

# Number of optimizer updates = training batches / grad accumulation
phase2_max_train_steps = 600
phase2_grad_accum = 4
phase2_num_optimizer_steps = math.ceil(phase2_max_train_steps / phase2_grad_accum)

phase2_scheduler_A = torch.optim.lr_scheduler.OneCycleLR(
    phase2_optimizer_A,
    max_lr=5e-4,
    total_steps=phase2_num_optimizer_steps,
    pct_start=0.10,
    anneal_strategy="cos",
)

print("Phase 2 max train batches:", phase2_max_train_steps)
print("Grad accumulation:", phase2_grad_accum)
print("Optimizer steps:", phase2_num_optimizer_steps)

phase2_history_A = train_phase2_vqa_with_replay(
    lm_model=lm_lora_A,
    connector=connector_A,
    vqa_loader=train_vqa_loader_A,
    replay_loader=alpaca_replay_loader,
    val_dataset=val_vqa_dataset_A,
    optimizer=phase2_optimizer_A,
    scheduler=phase2_scheduler_A,
    device=DEVICE,
    lambda_replay=0.2,
    num_epochs=1,
    grad_accum_steps=phase2_grad_accum,
    log_every=25,
    eval_every=300,
    max_train_steps=phase2_max_train_steps,
)

Plot Phase 2 losses

In [ ]:


plt.figure(figsize=(8, 4))
plt.plot(phase2_history_A["step"], phase2_history_A["vqa_loss"], label="VQA loss")
plt.plot(phase2_history_A["step"], phase2_history_A["replay_loss"], label="Replay loss")
plt.plot(phase2_history_A["step"], phase2_history_A["mixed_loss"], label="Mixed loss")
plt.xlabel("Training batch step")
plt.ylabel("Loss")
plt.title("Part A Phase 2 Losses")
plt.legend()
plt.grid(True)
plt.show()

if phase2_history_A["quick_vqa_acc"]:
    steps, accs = zip(*phase2_history_A["quick_vqa_acc"])
    plt.figure(figsize=(7, 4))
    plt.plot(steps, accs, marker="o")
    plt.xlabel("Training batch step")
    plt.ylabel("Quick VQA accuracy")
    plt.title("Part A Phase 2 Quick VQA Accuracy")
    plt.grid(True)
    plt.show()

if phase2_history_A["R"]:
    steps, Rs = zip(*phase2_history_A["R"])
    plt.figure(figsize=(7, 4))
    plt.plot(steps, Rs, marker="o")
    plt.axhline(1.0, linestyle="--")
    plt.xlabel("Training batch step")
    plt.ylabel("Forgetting ratio R")
    plt.title("Part A Phase 2 Forgetting Ratio")
    plt.grid(True)
    plt.show()

Save Phase 2 600-step pilot checkpoint

In [ ]:

phase2_pilot_dir_A = os.path.join(WEIGHTS_DIR, "phaseA2_pilot_600steps_lora")
os.makedirs(phase2_pilot_dir_A, exist_ok=True)

lm_lora_A.save_pretrained(phase2_pilot_dir_A)

phase2_pilot_connector_path_A = os.path.join(phase2_pilot_dir_A, "connector_phaseA2_pilot_600steps.pt")

phase2_pilot_payload_A = {
    "connector_state_dict": connector_A.state_dict(),
    "phase2_history": phase2_history_A,
    "lambda_replay": 0.2,
    "max_train_steps": 600,
    "PPL0": PPL0,
    "pilot_final_quick_vqa_acc": phase2_history_A["quick_vqa_acc"][-1] if phase2_history_A["quick_vqa_acc"] else None,
    "pilot_final_R": phase2_history_A["R"][-1] if phase2_history_A["R"] else None,
}

torch.save(phase2_pilot_payload_A, phase2_pilot_connector_path_A)

print("Saved Phase 2 pilot LoRA dir:")
print(phase2_pilot_dir_A)
print("Saved Phase 2 pilot connector:")
print(phase2_pilot_connector_path_A)
print("Contents:", os.listdir(phase2_pilot_dir_A))

Reload clean base LM and soft-rescaled connector for full Phase 2

In [ ]:
import gc

# Free current LoRA model from GPU memory
try:
    del lm_lora_A
except NameError:
    pass

try:
    del lm
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory after cleanup:", get_gpu_memory_gb())

# Reload base LM fresh
lm = AutoModelForCausalLM.from_pretrained(
    cfg.lm_name,
    torch_dtype=cfg.dtype,
    device_map=None,
)

lm.to(DEVICE)
lm.config.use_cache = False
lm.eval()
set_requires_grad(lm, False)

print("Reloaded clean base LM.")
print_parameter_summary(lm, "Clean frozen SmolLM2")

# Recreate connector and load soft-rescaled Phase 1 checkpoint
connector_A = MLPConnector(
    in_dim=partA_cfg.clip_hidden_size,
    hidden_dim=partA_cfg.lm_hidden_size,
    out_dim=partA_cfg.lm_hidden_size,
).to(DEVICE)

soft_payload = torch.load(phase1_soft_ckpt_path_A, map_location=DEVICE)
connector_A.load_state_dict(soft_payload["connector_state_dict"])
connector_A.to(DEVICE)
connector_A.train()
set_requires_grad(connector_A, True)

print("Reloaded soft-rescaled Phase 1 connector.")
print("Stored soft rnorm:", soft_payload["rnorm_phase1_soft_rescaled"])

# Verify rnorm again
rnorm_check_full_phase2 = compute_connector_rnorm(
    connector=connector_A,
    lm_model=lm,
    data_loader=phase1_caption_loader_A,
    tokenizer=tokenizer,
    device=DEVICE,
    num_batches=5,
)

print("Current connector rnorm before full Phase 2:")
print(rnorm_check_full_phase2)

Apply fresh LoRA for full Phase 2

In [ ]:

set_requires_grad(lm, False)

lora_config_A_full = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=list(cfg.lora_targets),
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

lm_lora_A = get_peft_model(lm, lora_config_A_full)
lm_lora_A.to(DEVICE)
lm_lora_A.train()

connector_A.train()
set_requires_grad(connector_A, True)

print_parameter_summary(lm_lora_A, "Fresh SmolLM2 LoRA for full Phase 2")
print_parameter_summary(connector_A, "Connector for full Phase 2")

try:
    lm_lora_A.print_trainable_parameters()
except Exception as e:
    print(e)

total_trainable_phase2_full = (
    sum(p.numel() for p in lm_lora_A.parameters() if p.requires_grad)
    + sum(p.numel() for p in connector_A.parameters() if p.requires_grad)
)

total_params_phase2_full = (
    sum(p.numel() for p in lm_lora_A.parameters())
    + sum(p.numel() for p in connector_A.parameters())
)

print("\nTotal trainable Phase 2 full params including connector:", f"{total_trainable_phase2_full:,}")
print("Trainable % including connector:", round(100 * total_trainable_phase2_full / total_params_phase2_full, 4))

Run full Phase 2 epoch

In [ ]:
phase2_grad_accum_full = 4
phase2_max_train_steps_full = len(train_vqa_loader_A)  # full epoch over 50,000 VQA pairs

phase2_num_optimizer_steps_full = math.ceil(
    phase2_max_train_steps_full / phase2_grad_accum_full
)

print("Full Phase 2 train batches:", phase2_max_train_steps_full)
print("Grad accumulation:", phase2_grad_accum_full)
print("Optimizer steps:", phase2_num_optimizer_steps_full)

phase2_trainable_params_full = [
    p for p in list(lm_lora_A.parameters()) + list(connector_A.parameters())
    if p.requires_grad
]

phase2_optimizer_A_full = torch.optim.AdamW(
    phase2_trainable_params_full,
    lr=5e-4,
    weight_decay=0.01,
)

phase2_scheduler_A_full = torch.optim.lr_scheduler.OneCycleLR(
    phase2_optimizer_A_full,
    max_lr=5e-4,
    total_steps=phase2_num_optimizer_steps_full,
    pct_start=0.10,
    anneal_strategy="cos",
)

phase2_history_A_full = train_phase2_vqa_with_replay(
    lm_model=lm_lora_A,
    connector=connector_A,
    vqa_loader=train_vqa_loader_A,
    replay_loader=alpaca_replay_loader,
    val_dataset=val_vqa_dataset_A,
    optimizer=phase2_optimizer_A_full,
    scheduler=phase2_scheduler_A_full,
    device=DEVICE,
    lambda_replay=0.2,
    num_epochs=1,
    grad_accum_steps=phase2_grad_accum_full,
    log_every=100,
    eval_every=600,
    max_train_steps=phase2_max_train_steps_full,
)

Plot full Phase 2 losses and metrics

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(phase2_history_A_full["step"], phase2_history_A_full["vqa_loss"], label="VQA loss")
plt.plot(phase2_history_A_full["step"], phase2_history_A_full["replay_loss"], label="Replay loss")
plt.plot(phase2_history_A_full["step"], phase2_history_A_full["mixed_loss"], label="Mixed loss")
plt.xlabel("Training batch step")
plt.ylabel("Loss")
plt.title("Part A Phase 2 Full Epoch Losses")
plt.legend()
plt.grid(True)
plt.show()

if phase2_history_A_full["quick_vqa_acc"]:
    steps, accs = zip(*phase2_history_A_full["quick_vqa_acc"])
    plt.figure(figsize=(7, 4))
    plt.plot(steps, accs, marker="o")
    plt.xlabel("Training batch step")
    plt.ylabel("Quick VQA accuracy")
    plt.title("Part A Phase 2 Full Epoch Quick VQA Accuracy")
    plt.grid(True)
    plt.show()

if phase2_history_A_full["R"]:
    steps, Rs = zip(*phase2_history_A_full["R"])
    plt.figure(figsize=(7, 4))
    plt.plot(steps, Rs, marker="o")
    plt.axhline(1.0, linestyle="--", label="No forgetting")
    plt.xlabel("Training batch step")
    plt.ylabel("Forgetting ratio R")
    plt.title("Part A Phase 2 Full Epoch Forgetting Ratio")
    plt.legend()
    plt.grid(True)
    plt.show()

Save full Phase 2 checkpoint

In [ ]:
phase2_full_dir_A = os.path.join(WEIGHTS_DIR, "phaseA2_full_lora")
os.makedirs(phase2_full_dir_A, exist_ok=True)

lm_lora_A.save_pretrained(phase2_full_dir_A)

phase2_full_connector_path_A = os.path.join(phase2_full_dir_A, "connector_phaseA2_full.pt")

phase2_full_payload_A = {
    "connector_state_dict": connector_A.state_dict(),
    "phase2_history": phase2_history_A_full,
    "lambda_replay": 0.2,
    "max_train_steps": phase2_max_train_steps_full,
    "grad_accum_steps": phase2_grad_accum_full,
    "PPL0": PPL0,
    "phase1_soft_ckpt_path": phase1_soft_ckpt_path_A,
    "partA_config": partA_cfg.__dict__,
    "global_config": cfg.__dict__,
}

torch.save(phase2_full_payload_A, phase2_full_connector_path_A)

print("Saved full Phase 2 LoRA adapter dir:")
print(phase2_full_dir_A)
print("Saved full Phase 2 connector:")
print(phase2_full_connector_path_A)
print("Directory contents:")
print(os.listdir(phase2_full_dir_A))

Full Phase 2 VQA evaluation helpers

In [ ]:

from collections import defaultdict, Counter
import pandas as pd

@torch.no_grad()
def generate_vqa_answer_partA_text_only(
    lm_model,
    question: str,
    tokenizer,
    device: torch.device,
    max_new_tokens: int = 6,
):
    """
    Text-only baseline:
        [BOS, question] -> answer
    No visual tokens, no connector.
    """
    lm_model.eval()

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    q_enc = tokenizer(
        [question],
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=48,
    )

    tokenizer.padding_side = old_padding_side

    input_embeddings = get_lm_input_embeddings(lm_model)

    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
    bos_ids = torch.full((1, 1), bos_id, dtype=torch.long, device=device)
    bos_embeds = input_embeddings(bos_ids)

    q_ids = q_enc["input_ids"].to(device)
    q_embeds = input_embeddings(q_ids)

    inputs_embeds = torch.cat([bos_embeds, q_embeds], dim=1)
    attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)

    generated_ids = []

    for _ in range(max_new_tokens):
        outputs = lm_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            use_cache=False,
        )

        logits = outputs.logits[:, -1, :]
        next_id = torch.argmax(logits, dim=-1)

        if next_id.item() == tokenizer.eos_token_id:
            break

        generated_ids.append(next_id.item())

        next_embed = input_embeddings(next_id[:, None])
        inputs_embeds = torch.cat([inputs_embeds, next_embed], dim=1)

        next_attn = torch.ones((1, 1), dtype=torch.long, device=device)
        attention_mask = torch.cat([attention_mask, next_attn], dim=1)

    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return raw.strip(), normalize_answer(raw)


def compute_majority_baselines(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Computes global majority and per-template majority baselines.
    """
    answers = [normalize_answer(r["answer"]) for r in records]
    global_majority = Counter(answers).most_common(1)[0][0]

    global_correct = sum(a == global_majority for a in answers)
    global_acc = global_correct / len(answers)

    template_majority = {}
    template_accs = {}

    grouped = defaultdict(list)
    for r in records:
        grouped[r["template_id"]].append(normalize_answer(r["answer"]))

    for template_id, ans_list in grouped.items():
        maj = Counter(ans_list).most_common(1)[0][0]
        template_majority[template_id] = maj
        template_accs[template_id] = sum(a == maj for a in ans_list) / len(ans_list)

    return {
        "global_majority_answer": global_majority,
        "global_majority_acc": global_acc,
        "template_majority_answers": template_majority,
        "template_majority_accs": template_accs,
    }


@torch.no_grad()
def evaluate_vqa_partA_full(
    lm_model,
    connector,
    dataset,
    max_examples: Optional[int] = None,
    desc: str = "Full VQA eval",
):
    """
    Exact-match generated-answer evaluation.
    """
    lm_model.eval()
    connector.eval()

    n = len(dataset) if max_examples is None else min(max_examples, len(dataset))

    rows = []
    correct = 0

    for i in tqdm(range(n), desc=f"{desc} ({n})"):
        item = dataset[i]

        raw_pred, norm_pred = generate_vqa_answer_partA(
            lm_model=lm_model,
            connector=connector,
            clip_features=item["clip_features"],
            question=item["question"],
            tokenizer=tokenizer,
            device=DEVICE,
            max_new_tokens=6,
        )

        gold = normalize_answer(item["answer"])
        is_correct = norm_pred == gold
        correct += int(is_correct)

        rows.append({
            "i": i,
            "question": item["question"],
            "gold": gold,
            "raw_pred": raw_pred,
            "norm_pred": norm_pred,
            "correct": is_correct,
            "template_id": item["template_id"],
            "class_name": item["class_name"],
            "local_id": item["local_id"],
        })

    df = pd.DataFrame(rows)

    overall_acc = df["correct"].mean()

    per_template = (
        df.groupby("template_id")["correct"]
        .agg(["mean", "count"])
        .sort_index()
        .reset_index()
        .rename(columns={"mean": "accuracy"})
    )

    per_class = (
        df.groupby("class_name")["correct"]
        .agg(["mean", "count"])
        .sort_index()
        .reset_index()
        .rename(columns={"mean": "accuracy"})
    )

    return {
        "overall_acc": overall_acc,
        "per_template": per_template,
        "per_class": per_class,
        "df": df,
    }


@torch.no_grad()
def evaluate_text_only_baseline_partA(
    lm_model,
    dataset,
    max_examples: int = 1000,
):
    """
    Text-only baseline on a subset.
    """
    lm_model.eval()

    n = min(max_examples, len(dataset))
    rows = []
    correct = 0

    for i in tqdm(range(n), desc=f"Text-only baseline ({n})"):
        item = dataset[i]

        raw_pred, norm_pred = generate_vqa_answer_partA_text_only(
            lm_model=lm_model,
            question=item["question"],
            tokenizer=tokenizer,
            device=DEVICE,
            max_new_tokens=6,
        )

        gold = normalize_answer(item["answer"])
        is_correct = norm_pred == gold
        correct += int(is_correct)

        rows.append({
            "i": i,
            "question": item["question"],
            "gold": gold,
            "raw_pred": raw_pred,
            "norm_pred": norm_pred,
            "correct": is_correct,
            "template_id": item["template_id"],
            "class_name": item["class_name"],
        })

    df = pd.DataFrame(rows)

    return {
        "overall_acc": correct / n,
        "df": df,
        "per_template": (
            df.groupby("template_id")["correct"]
            .agg(["mean", "count"])
            .sort_index()
            .reset_index()
            .rename(columns={"mean": "accuracy"})
        ),
    }

Run full Phase 2 evaluation

In [ ]:

reset_peak_memory()

phase2_full_eval_A = evaluate_vqa_partA_full(
    lm_model=lm_lora_A,
    connector=connector_A,
    dataset=val_vqa_dataset_A,
    max_examples=None,  # full 10,000 validation pairs
    desc="Phase 2 full validation",
)

print("\nPhase 2 full validation overall accuracy:")
print(phase2_full_eval_A["overall_acc"])

print("\nPer-template accuracy:")
display(phase2_full_eval_A["per_template"])

print("\nPer-class accuracy:")
display(phase2_full_eval_A["per_class"])

print("\nGPU memory after full eval:")
print(get_gpu_memory_gb())

Majority and text-only baselines

In [ ]:

majority_baselines_A = compute_majority_baselines(val_vqa_records_A)

print("Majority baseline:")
print(json.dumps(majority_baselines_A, indent=2))

# Text-only baseline on 1,000 examples first
text_only_eval_A = evaluate_text_only_baseline_partA(
    lm_model=lm_lora_A,
    dataset=val_vqa_dataset_A,
    max_examples=1000,
)

print("\nText-only baseline overall accuracy on 1,000 examples:")
print(text_only_eval_A["overall_acc"])

print("\nText-only per-template accuracy:")
display(text_only_eval_A["per_template"])

Save Phase 2 evaluation results

In [ ]:
phase2_eval_dir_A = os.path.join(OUTPUTS_DIR, "partA_phase2_eval")
os.makedirs(phase2_eval_dir_A, exist_ok=True)

phase2_full_eval_A["df"].to_csv(
    os.path.join(phase2_eval_dir_A, "phase2_full_val_predictions.csv"),
    index=False,
)

phase2_full_eval_A["per_template"].to_csv(
    os.path.join(phase2_eval_dir_A, "phase2_per_template.csv"),
    index=False,
)

phase2_full_eval_A["per_class"].to_csv(
    os.path.join(phase2_eval_dir_A, "phase2_per_class.csv"),
    index=False,
)

text_only_eval_A["df"].to_csv(
    os.path.join(phase2_eval_dir_A, "phase2_text_only_1000_predictions.csv"),
    index=False,
)

with open(os.path.join(phase2_eval_dir_A, "phase2_summary.json"), "w") as f:
    json.dump(
        {
            "phase2_full_overall_acc": float(phase2_full_eval_A["overall_acc"]),
            "majority_baselines": majority_baselines_A,
            "text_only_1000_acc": float(text_only_eval_A["overall_acc"]),
            "PPL0": float(PPL0),
            "phase2_eval_R_last": phase2_history_A_full["R"][-1] if phase2_history_A_full["R"] else None,
        },
        f,
        indent=2,
    )

print("Saved Phase 2 evaluation outputs to:")
print(phase2_eval_dir_A)
print("Files:")
print(os.listdir(phase2_eval_dir_A))

Helper to reset Phase 2 model for each λ

In [ ]:
import gc

def reset_partA_phase2_model_for_ablation():
    """
    Reloads clean base LM, soft-rescaled Phase 1 connector, and fresh LoRA.
    This ensures every lambda ablation starts from the same initialization.
    """
    global lm, lm_lora_A, connector_A

    try:
        del lm_lora_A
    except Exception:
        pass

    try:
        del lm
    except Exception:
        pass

    try:
        del connector_A
    except Exception:
        pass

    gc.collect()
    torch.cuda.empty_cache()

    # Reload clean base LM
    lm = AutoModelForCausalLM.from_pretrained(
        cfg.lm_name,
        torch_dtype=cfg.dtype,
        device_map=None,
    )

    lm.to(DEVICE)
    lm.config.use_cache = False
    lm.eval()
    set_requires_grad(lm, False)

    # Reload soft-rescaled connector
    connector_A = MLPConnector(
        in_dim=partA_cfg.clip_hidden_size,
        hidden_dim=partA_cfg.lm_hidden_size,
        out_dim=partA_cfg.lm_hidden_size,
    ).to(DEVICE)

    soft_payload = torch.load(phase1_soft_ckpt_path_A, map_location=DEVICE)
    connector_A.load_state_dict(soft_payload["connector_state_dict"])
    connector_A.train()
    set_requires_grad(connector_A, True)

    # Fresh LoRA
    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        target_modules=list(cfg.lora_targets),
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    lm_lora_A = get_peft_model(lm, lora_config)
    lm_lora_A.to(DEVICE)
    lm_lora_A.train()

    return lm_lora_A, connector_A

Run compact λ ablations

In [ ]:
lambda_ablation_values = [0.00, 0.05, 0.50]

phase2_ablation_results_A = {}

for lam in lambda_ablation_values:
    print("\n" + "=" * 80)
    print(f"Running Part A Phase 2 lambda ablation: lambda = {lam}")
    print("=" * 80)

    lm_lora_A, connector_A = reset_partA_phase2_model_for_ablation()

    ablation_max_steps = 600
    ablation_grad_accum = 4
    ablation_optimizer_steps = math.ceil(ablation_max_steps / ablation_grad_accum)

    trainable_params = [
        p for p in list(lm_lora_A.parameters()) + list(connector_A.parameters())
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=5e-4,
        weight_decay=0.01,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=5e-4,
        total_steps=ablation_optimizer_steps,
        pct_start=0.10,
        anneal_strategy="cos",
    )

    history = train_phase2_vqa_with_replay(
        lm_model=lm_lora_A,
        connector=connector_A,
        vqa_loader=train_vqa_loader_A,
        replay_loader=alpaca_replay_loader,
        val_dataset=val_vqa_dataset_A,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        lambda_replay=lam,
        num_epochs=1,
        grad_accum_steps=ablation_grad_accum,
        log_every=100,
        eval_every=600,
        max_train_steps=ablation_max_steps,
    )

    final_quick_acc = history["quick_vqa_acc"][-1][1] if history["quick_vqa_acc"] else None
    final_ppl = history["ppl_replay"][-1][1] if history["ppl_replay"] else None
    final_R = history["R"][-1][1] if history["R"] else None

    phase2_ablation_results_A[lam] = {
        "lambda": lam,
        "steps": ablation_max_steps,
        "history": history,
        "quick_vqa_acc": final_quick_acc,
        "ppl": final_ppl,
        "R": final_R,
    }

    print("\nAblation result:")
    print({
        "lambda": lam,
        "quick_vqa_acc": final_quick_acc,
        "ppl": final_ppl,
        "R": final_R,
    })

Add existing λ=0.20 pilot and make ablation table

In [ ]:
# Add the already-run 600-step baseline pilot for lambda = 0.20
phase2_ablation_results_A[0.20] = {
    "lambda": 0.20,
    "steps": 600,
    "history": phase2_history_A,
    "quick_vqa_acc": phase2_history_A["quick_vqa_acc"][-1][1],
    "ppl": phase2_history_A["ppl_replay"][-1][1],
    "R": phase2_history_A["R"][-1][1],
}

ablation_rows = []

condition_names = {
    0.00: "No replay",
    0.05: "Weak replay",
    0.20: "Baseline",
    0.50: "Strong replay",
}

for lam in [0.00, 0.05, 0.20, 0.50]:
    result = phase2_ablation_results_A[lam]

    ablation_rows.append({
        "condition": condition_names[lam],
        "lambda": lam,
        "steps": result["steps"],
        "quick_vqa_acc": result["quick_vqa_acc"],
        "ppl": result["ppl"],
        "R": result["R"],
    })

phase2_lambda_ablation_df_A = pd.DataFrame(ablation_rows)

display(phase2_lambda_ablation_df_A)

# Save
ablation_dir_A = os.path.join(OUTPUTS_DIR, "partA_phase2_lambda_ablation")
os.makedirs(ablation_dir_A, exist_ok=True)

phase2_lambda_ablation_df_A.to_csv(
    os.path.join(ablation_dir_A, "phase2_lambda_ablation_600steps.csv"),
    index=False,
)

torch.save(
    phase2_ablation_results_A,
    os.path.join(ablation_dir_A, "phase2_lambda_ablation_histories.pt"),
)

print("Saved lambda ablation outputs to:")
print(ablation_dir_A)
print(os.listdir(ablation_dir_A))

## Part A Phase 2: Results and Replay Ablation

Phase 2 fine-tuned the MLP connector and LoRA adapters using the mixed objective  
`L = L_VQA + λ L_LM`, while keeping the CLIP encoder and base SmolLM2 weights frozen. The full Phase 2 run with `λ = 0.20` achieved **97.16% exact-match accuracy** on the full 10,000-pair validation set. Performance was high across templates: `can_fly = 96.90%`, `class_presence = 97.05%`, `is_animal = 99.55%`, `object_recognition = 92.80%`, and `vehicle_or_living = 99.50%`. Object recognition is the hardest template because it requires choosing among all ten CIFAR-10 classes, while the other templates are binary or coarse-grained.

The majority baseline achieved only **34.0%**, and the text-only baseline achieved **33.7%** on 1,000 examples. This confirms that the trained model is not solving the task from question priors alone; it is using the CLIP-derived visual tokens.

The replay-weight ablation shows the purpose of language replay. With no replay (`λ = 0`), quick VQA accuracy remained high at **95.5%**, but the forgetting ratio increased to **R = 1.075**, meaning language perplexity became worse than the original model. Adding replay reduced R substantially: `λ = 0.05` gave `R = 0.776`, `λ = 0.20` gave `R = 0.633`, and `λ = 0.50` gave `R = 0.589`. Thus, replay protects the LM’s language behavior while still allowing visual alignment. In this small Alpaca evaluation subset, replay not only prevented forgetting but improved measured replay perplexity relative to PPL0.

Reload full Phase 2 checkpoint for Phase 3

In [ ]:

import gc
from peft import PeftModel

# Clean current model state, which may be from the last lambda ablation.
try:
    del lm_lora_A
except Exception:
    pass

try:
    del lm
except Exception:
    pass

try:
    del connector_A
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory after cleanup:", get_gpu_memory_gb())

# Reload clean frozen base LM
lm = AutoModelForCausalLM.from_pretrained(
    cfg.lm_name,
    torch_dtype=cfg.dtype,
    device_map=None,
)

lm.to(DEVICE)
lm.config.use_cache = False
set_requires_grad(lm, False)
lm.eval()

print("Reloaded clean base LM.")
print_parameter_summary(lm, "Clean frozen SmolLM2 before Phase 3")

# Load Phase 2 LoRA adapter as trainable
phase2_full_dir_A = os.path.join(WEIGHTS_DIR, "phaseA2_full_lora")
assert os.path.exists(phase2_full_dir_A), f"Missing Phase 2 adapter dir: {phase2_full_dir_A}"

lm_lora_A = PeftModel.from_pretrained(
    lm,
    phase2_full_dir_A,
    is_trainable=True,
)

lm_lora_A.to(DEVICE)
lm_lora_A.train()
lm_lora_A.config.use_cache = False

# Reload Phase 2 connector
phase2_full_connector_path_A = os.path.join(
    phase2_full_dir_A,
    "connector_phaseA2_full.pt",
)

assert os.path.exists(phase2_full_connector_path_A), f"Missing connector: {phase2_full_connector_path_A}"

phase2_full_payload_A = torch.load(
    phase2_full_connector_path_A,
    map_location=DEVICE,
)

connector_A = MLPConnector(
    in_dim=partA_cfg.clip_hidden_size,
    hidden_dim=partA_cfg.lm_hidden_size,
    out_dim=partA_cfg.lm_hidden_size,
).to(DEVICE)

connector_A.load_state_dict(phase2_full_payload_A["connector_state_dict"])
connector_A.train()
set_requires_grad(connector_A, True)

print("\nReloaded Phase 2 full adapter + connector.")
print_parameter_summary(lm_lora_A, "Phase 2 LoRA loaded for Phase 3")
print_parameter_summary(connector_A, "Phase 2 connector loaded for Phase 3")

try:
    lm_lora_A.print_trainable_parameters()
except Exception as e:
    print(e)

# Quick rnorm check
rnorm_before_phase3 = compute_connector_rnorm(
    connector=connector_A,
    lm_model=lm_lora_A,
    data_loader=phase1_caption_loader_A,
    tokenizer=tokenizer,
    device=DEVICE,
    num_batches=5,
)

print("\nConnector rnorm before Phase 3:")
print(rnorm_before_phase3)

Build 10,000-pair Phase 3 training subset

In [ ]:

phase3_train_size_A = 10_000

# The training records are generated as 5 VQA pairs per image.
# Taking the first 10,000 gives 2,000 images × 5 templates = balanced templates.
phase3_indices_A = list(range(phase3_train_size_A))

phase3_train_dataset_A = torch.utils.data.Subset(
    train_vqa_dataset_A,
    phase3_indices_A,
)

phase3_train_loader_A = DataLoader(
    phase3_train_dataset_A,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_partA_vqa_batch(batch, tokenizer),
)

# Check template balance
phase3_template_counts = Counter()
phase3_class_counts = Counter()

for idx in phase3_indices_A:
    rec = train_vqa_records_A[idx]
    phase3_template_counts[rec["template_id"]] += 1
    phase3_class_counts[rec["class_name"]] += 1

print("Phase 3 train size:", len(phase3_train_dataset_A))
print("Template counts:", dict(phase3_template_counts))
print("Class counts:", dict(phase3_class_counts))

batch = next(iter(phase3_train_loader_A))
print("\nPhase 3 batch clip_features:", batch["clip_features"].shape)
print("Phase 3 question ids:", batch["question_input_ids"].shape)
print("Phase 3 answer ids:", batch["answer_input_ids"].shape)
print("Sample Q:", batch["questions"][0])
print("Sample A:", batch["answers"][0])

assert len(phase3_train_dataset_A) == 10_000
assert all(v == 2000 for v in phase3_template_counts.values())

print("\nPhase 3 subset sanity check passed.")

Phase 3 training loop: VQA only, no replay

In [ ]:
def train_phase3_vqa_only(
    lm_model,
    connector,
    train_loader,
    val_dataset,
    optimizer,
    device: torch.device,
    num_epochs: int = 1,
    grad_accum_steps: int = 4,
    log_every: int = 50,
    eval_every: int = 300,
):
    """
    Phase 3:
        Load Phase 2 checkpoint.
        Train connector + LoRA on VQA only.
        No Alpaca replay, so lambda = 0.

    We track:
        VQA loss
        quick VQA accuracy
        replay PPL and R
    """
    lm_model.train()
    connector.train()

    scaler = torch.amp.GradScaler("cuda")

    history = {
        "step": [],
        "vqa_loss": [],
        "lr": [],
        "quick_vqa_acc": [],
        "ppl_replay": [],
        "R": [],
    }

    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    reset_peak_memory()
    start_time = time.time()

    total_batches = len(train_loader) * num_epochs
    pbar = tqdm(total=total_batches, desc="Phase 3 VQA-only training")

    for epoch in range(num_epochs):
        for batch_idx, vqa_batch in enumerate(train_loader):
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                inputs_embeds, attention_mask, labels = build_phase2_vqa_inputs_and_labels(
                    lm_model=lm_model,
                    connector=connector,
                    clip_features=vqa_batch["clip_features"],
                    question_input_ids=vqa_batch["question_input_ids"],
                    question_attention_mask=vqa_batch["question_attention_mask"],
                    answer_input_ids=vqa_batch["answer_input_ids"],
                    answer_attention_mask=vqa_batch["answer_attention_mask"],
                    device=device,
                )

                outputs = lm_model(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    labels=labels,
                    use_cache=False,
                )

                vqa_loss = outputs.loss
                loss_for_backward = vqa_loss / grad_accum_steps

            scaler.scale(loss_for_backward).backward()

            if (global_step + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)

                trainable_params = [
                    p for p in list(lm_model.parameters()) + list(connector.parameters())
                    if p.requires_grad and p.grad is not None
                ]

                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            global_step += 1
            pbar.update(1)

            if global_step % log_every == 0:
                lr = optimizer.param_groups[0]["lr"]

                print(
                    f"step={global_step:04d} | "
                    f"vqa={vqa_loss.item():.4f} | "
                    f"lr={lr:.2e} | "
                    f"mem={get_gpu_memory_gb()}"
                )

                history["step"].append(global_step)
                history["vqa_loss"].append(float(vqa_loss.detach().cpu()))
                history["lr"].append(lr)

            if global_step % eval_every == 0:
                quick_acc, _ = evaluate_vqa_partA_small(
                    lm_model=lm_model,
                    connector=connector,
                    dataset=val_dataset,
                    max_examples=200,
                )

                ppl_fine = compute_perplexity(
                    model=lm_model,
                    tokenizer=tokenizer,
                    texts=ppl0_texts,
                    device=device,
                    batch_size=4,
                    max_length=cfg.max_text_length,
                    desc=f"Phase 3 replay PPL step {global_step}",
                )

                R = ppl_fine / PPL0

                print(
                    f"\n[PHASE 3 EVAL step={global_step}] "
                    f"quick_vqa_acc={quick_acc:.4f} | "
                    f"PPL={ppl_fine:.4f} | "
                    f"R={R:.4f}\n"
                )

                history["quick_vqa_acc"].append((global_step, quick_acc))
                history["ppl_replay"].append((global_step, ppl_fine))
                history["R"].append((global_step, R))

                lm_model.train()
                connector.train()

    pbar.close()

    elapsed = time.time() - start_time

    print("\nPhase 3 training complete.")
    print("Elapsed minutes:", round(elapsed / 60, 2))
    print("Peak GPU memory GB:", get_gpu_memory_gb()["max_allocated"])

    return history

Run Phase 3

In [ ]:
phase3_trainable_params_A = [
    p for p in list(lm_lora_A.parameters()) + list(connector_A.parameters())
    if p.requires_grad
]

phase3_optimizer_A = torch.optim.AdamW(
    phase3_trainable_params_A,
    lr=2e-4,
    weight_decay=0.01,
)

print("Phase 3 train batches:", len(phase3_train_loader_A))
print("Grad accum:", 4)
print("Approx optimizer steps:", math.ceil(len(phase3_train_loader_A) / 4))

phase3_history_A = train_phase3_vqa_only(
    lm_model=lm_lora_A,
    connector=connector_A,
    train_loader=phase3_train_loader_A,
    val_dataset=val_vqa_dataset_A,
    optimizer=phase3_optimizer_A,
    device=DEVICE,
    num_epochs=1,
    grad_accum_steps=4,
    log_every=50,
    eval_every=300,
)

Plot Phase 3 quick metrics

In [ ]:

plt.figure(figsize=(7, 4))
plt.plot(phase3_history_A["step"], phase3_history_A["vqa_loss"], marker="o")
plt.xlabel("Training batch step")
plt.ylabel("VQA loss")
plt.title("Part A Phase 3 VQA-only Loss")
plt.grid(True)
plt.show()

if phase3_history_A["quick_vqa_acc"]:
    steps, accs = zip(*phase3_history_A["quick_vqa_acc"])
    plt.figure(figsize=(7, 4))
    plt.plot(steps, accs, marker="o")
    plt.xlabel("Training batch step")
    plt.ylabel("Quick VQA accuracy")
    plt.title("Part A Phase 3 Quick VQA Accuracy")
    plt.grid(True)
    plt.show()

if phase3_history_A["R"]:
    steps, Rs = zip(*phase3_history_A["R"])
    plt.figure(figsize=(7, 4))
    plt.plot(steps, Rs, marker="o")
    plt.axhline(1.0, linestyle="--", label="No forgetting")
    plt.xlabel("Training batch step")
    plt.ylabel("Forgetting ratio R")
    plt.title("Part A Phase 3 Forgetting Ratio")
    plt.legend()
    plt.grid(True)
    plt.show()

Save Phase 3 checkpoint

In [ ]:
phase3_dir_A = os.path.join(WEIGHTS_DIR, "phaseA3_vqa_only_lora")
os.makedirs(phase3_dir_A, exist_ok=True)

lm_lora_A.save_pretrained(phase3_dir_A)

phase3_connector_path_A = os.path.join(phase3_dir_A, "connector_phaseA3.pt")

phase3_payload_A = {
    "connector_state_dict": connector_A.state_dict(),
    "phase3_history": phase3_history_A,
    "phase2_full_dir": phase2_full_dir_A,
    "phase2_full_connector_path": phase2_full_connector_path_A,
    "phase3_train_size": phase3_train_size_A,
    "lr": 2e-4,
    "lambda_replay": 0.0,
    "PPL0": PPL0,
    "partA_config": partA_cfg.__dict__,
    "global_config": cfg.__dict__,
}

torch.save(phase3_payload_A, phase3_connector_path_A)

print("Saved Phase 3 LoRA adapter dir:")
print(phase3_dir_A)
print("Saved Phase 3 connector:")
print(phase3_connector_path_A)
print("Directory contents:")
print(os.listdir(phase3_dir_A))

## Part A Phase 3: VQA-Only Alignment

Phase 3 started from the full Phase 2 checkpoint and continued training the connector and LoRA adapters on 10,000 VQA pairs with no language replay (`λ = 0`). The goal was to test whether removing replay and focusing only on VQA improves image-grounded accuracy, and whether it increases forgetting.

The Phase 3 subset was exactly balanced across the five VQA templates, with 2,000 examples per template. Class counts were approximately but not perfectly balanced because the subset was selected deterministically from the shuffled VQA training records.

Phase 3 training was stable. VQA loss stayed low, quick validation accuracy was `96.0%` at step 300 and `95.5%` at step 600, and replay forgetting ratio stayed around `R ≈ 0.53`. Compared with Phase 2, Phase 3 did not improve quick VQA accuracy; in fact, it slightly decreased from the best Phase 2 quick result. Forgetting also did not increase noticeably, probably because Phase 3 was short and started from a replay-protected Phase 2 model.

The conclusion is that Phase 2 already learned the required VQA alignment well. Additional VQA-only tuning was not beneficial in this run, so the Phase 2 checkpoint is the stronger main model.

Full Phase 3 validation

In [ ]:

reset_peak_memory()

phase3_full_eval_A = evaluate_vqa_partA_full(
    lm_model=lm_lora_A,
    connector=connector_A,
    dataset=val_vqa_dataset_A,
    max_examples=None,
    desc="Phase 3 full validation",
)

print("\nPhase 3 full validation overall accuracy:")
print(phase3_full_eval_A["overall_acc"])

print("\nPhase 3 per-template accuracy:")
display(phase3_full_eval_A["per_template"])

print("\nPhase 3 per-class accuracy:")
display(phase3_full_eval_A["per_class"])

print("\nGPU memory after Phase 3 full eval:")
print(get_gpu_memory_gb())

Compute Phase 3 replay PPL and R

In [ ]:
phase3_PPL = compute_perplexity(
    model=lm_lora_A,
    tokenizer=tokenizer,
    texts=ppl0_texts,
    device=DEVICE,
    batch_size=4,
    max_length=cfg.max_text_length,
    desc="Phase 3 final replay PPL",
)

phase3_R = phase3_PPL / PPL0

print("PPL0:", PPL0)
print("Phase 3 PPL:", phase3_PPL)
print("Phase 3 R:", phase3_R)

Build final Phase 2 vs Phase 3 comparison table

In [ ]:
phase2_full_overall_acc = 0.9716
phase2_final_PPL = 4.1436
phase2_final_R = 0.5280

phase2_quick_best = 0.9750
phase2_quick_final = 0.9700

phase3_quick_step300 = phase3_history_A["quick_vqa_acc"][0][1] if len(phase3_history_A["quick_vqa_acc"]) > 0 else None
phase3_quick_final = phase3_history_A["quick_vqa_acc"][-1][1] if len(phase3_history_A["quick_vqa_acc"]) > 0 else None

phase_comparison_A = pd.DataFrame([
    {
        "phase": "Phase 2: VQA + replay",
        "training_objective": "L_VQA + 0.2 L_LM",
        "train_vqa_pairs": 50_000,
        "replay": "Yes",
        "quick_vqa_final": phase2_quick_final,
        "full_val_acc": phase2_full_overall_acc,
        "PPL": phase2_final_PPL,
        "R": phase2_final_R,
    },
    {
        "phase": "Phase 3: VQA only",
        "training_objective": "L_VQA",
        "train_vqa_pairs": 10_000,
        "replay": "No",
        "quick_vqa_final": phase3_quick_final,
        "full_val_acc": float(phase3_full_eval_A["overall_acc"]),
        "PPL": float(phase3_PPL),
        "R": float(phase3_R),
    },
])

display(phase_comparison_A)

In [ ]:

phase3_eval_dir_A = os.path.join(OUTPUTS_DIR, "partA_phase3_eval")
os.makedirs(phase3_eval_dir_A, exist_ok=True)

phase3_full_eval_A["df"].to_csv(
    os.path.join(phase3_eval_dir_A, "phase3_full_val_predictions.csv"),
    index=False,
)

phase3_full_eval_A["per_template"].to_csv(
    os.path.join(phase3_eval_dir_A, "phase3_per_template.csv"),
    index=False,
)

phase3_full_eval_A["per_class"].to_csv(
    os.path.join(phase3_eval_dir_A, "phase3_per_class.csv"),
    index=False,
)

phase_comparison_A.to_csv(
    os.path.join(phase3_eval_dir_A, "phase2_vs_phase3_comparison.csv"),
    index=False,
)

with open(os.path.join(phase3_eval_dir_A, "phase3_summary.json"), "w") as f:
    json.dump(
        {
            "phase3_full_overall_acc": float(phase3_full_eval_A["overall_acc"]),
            "phase3_PPL": float(phase3_PPL),
            "phase3_R": float(phase3_R),
            "phase2_full_overall_acc": float(phase2_full_overall_acc),
            "phase2_PPL": float(phase2_final_PPL),
            "phase2_R": float(phase2_final_R),
            "phase3_quick_final": float(phase3_quick_final),
            "rnorm_before_phase3": rnorm_before_phase3,
        },
        f,
        indent=2,
    )

print("Saved Phase 3 evaluation outputs to:")
print(phase3_eval_dir_A)
print("Files:")
print(os.listdir(phase3_eval_dir_A))

Phase 0–3 VQA accuracy and R summary plots

In [ ]:
import gc
from peft import PeftModel

@torch.no_grad()
def compute_phase1_quick_vqa_acc():
    """
    Reloads base frozen LM + soft-rescaled Phase 1 connector
    and evaluates VQA on the fixed first 200 val examples.

    Phase 1 was trained only on captions, not VQA, so this measures
    how much VQA ability exists before answer-supervised alignment.
    """
    global lm_phase1_tmp, connector_phase1_tmp

    gc.collect()
    torch.cuda.empty_cache()

    lm_phase1_tmp = AutoModelForCausalLM.from_pretrained(
        cfg.lm_name,
        torch_dtype=cfg.dtype,
        device_map=None,
    ).to(DEVICE)

    lm_phase1_tmp.config.use_cache = False
    set_requires_grad(lm_phase1_tmp, False)
    lm_phase1_tmp.eval()

    connector_phase1_tmp = MLPConnector(
        in_dim=partA_cfg.clip_hidden_size,
        hidden_dim=partA_cfg.lm_hidden_size,
        out_dim=partA_cfg.lm_hidden_size,
    ).to(DEVICE)

    phase1_payload = torch.load(phase1_soft_ckpt_path_A, map_location=DEVICE)
    connector_phase1_tmp.load_state_dict(phase1_payload["connector_state_dict"])
    connector_phase1_tmp.eval()
    set_requires_grad(connector_phase1_tmp, False)

    acc, rows = evaluate_vqa_partA_small(
        lm_model=lm_phase1_tmp,
        connector=connector_phase1_tmp,
        dataset=val_vqa_dataset_A,
        max_examples=200,
    )

    del lm_phase1_tmp
    del connector_phase1_tmp
    gc.collect()
    torch.cuda.empty_cache()

    return acc, rows


# Phase 0 text-only baseline on first 200 examples from earlier 1000-example baseline if available.
if "text_only_eval_A" in globals():
    phase0_quick_acc = text_only_eval_A["df"].iloc[:200]["correct"].mean()
else:
    text_only_200 = evaluate_text_only_baseline_partA(
        lm_model=lm_lora_A,
        dataset=val_vqa_dataset_A,
        max_examples=200,
    )
    phase0_quick_acc = text_only_200["overall_acc"]

phase1_quick_acc, phase1_quick_rows = compute_phase1_quick_vqa_acc()

phase2_quick_acc = phase2_history_A_full["quick_vqa_acc"][-1][1]
phase3_quick_acc = phase3_history_A["quick_vqa_acc"][-1][1]

# R values
phase0_R = 1.0
phase1_R = 1.0  # LM was frozen in Phase 1, so pure-text PPL remains the base LM PPL.
phase2_R = phase2_final_R
phase3_R_final = phase3_R

phase_summary_A = pd.DataFrame([
    {
        "phase": "Phase 0: text-only/base",
        "quick_vqa_acc_200": phase0_quick_acc,
        "R": phase0_R,
        "notes": "No visual tokens; text-only baseline.",
    },
    {
        "phase": "Phase 1: connector caption init",
        "quick_vqa_acc_200": phase1_quick_acc,
        "R": phase1_R,
        "notes": "Connector trained on captions only; LM frozen.",
    },
    {
        "phase": "Phase 2: VQA + replay",
        "quick_vqa_acc_200": phase2_quick_acc,
        "R": phase2_R,
        "notes": "Connector + LoRA; mixed VQA and Alpaca replay.",
    },
    {
        "phase": "Phase 3: VQA only",
        "quick_vqa_acc_200": phase3_quick_acc,
        "R": phase3_R_final,
        "notes": "Connector + LoRA; VQA-only alignment.",
    },
])

display(phase_summary_A)

# Plot VQA accuracy
plt.figure(figsize=(9, 4))
plt.plot(
    phase_summary_A["phase"],
    phase_summary_A["quick_vqa_acc_200"],
    marker="o",
)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Quick VQA accuracy on fixed 200 val pairs")
plt.title("Part A VQA Accuracy Across Phases 0–3")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot R
plt.figure(figsize=(9, 4))
plt.plot(
    phase_summary_A["phase"],
    phase_summary_A["R"],
    marker="o",
)
plt.axhline(1.0, linestyle="--", label="No forgetting threshold")
plt.xticks(rotation=25, ha="right")
plt.ylabel("Forgetting ratio R = PPLfine / PPL0")
plt.title("Part A Forgetting Ratio Across Phases 0–3")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Reload best Phase 2 model for qualitative examples

In [ ]:
def load_phase2_full_model_for_eval():
    """
    Loads the main/best Part A Phase 2 model:
        clean base LM + Phase 2 LoRA + Phase 2 connector.
    """
    gc.collect()
    torch.cuda.empty_cache()

    base_lm = AutoModelForCausalLM.from_pretrained(
        cfg.lm_name,
        torch_dtype=cfg.dtype,
        device_map=None,
    ).to(DEVICE)

    base_lm.config.use_cache = False
    set_requires_grad(base_lm, False)
    base_lm.eval()

    phase2_lora = PeftModel.from_pretrained(
        base_lm,
        phase2_full_dir_A,
        is_trainable=False,
    ).to(DEVICE)

    phase2_lora.eval()

    phase2_connector = MLPConnector(
        in_dim=partA_cfg.clip_hidden_size,
        hidden_dim=partA_cfg.lm_hidden_size,
        out_dim=partA_cfg.lm_hidden_size,
    ).to(DEVICE)

    payload = torch.load(phase2_full_connector_path_A, map_location=DEVICE)
    phase2_connector.load_state_dict(payload["connector_state_dict"])
    phase2_connector.eval()
    set_requires_grad(phase2_connector, False)

    return phase2_lora, phase2_connector


lm_eval_A, connector_eval_A = load_phase2_full_model_for_eval()

print("Loaded Phase 2 model for qualitative evaluation.")
print_parameter_summary(lm_eval_A, "Phase 2 LoRA eval model")
print_parameter_summary(connector_eval_A, "Phase 2 connector eval")

Qualitative examples with top-5 logits

In [ ]:
@torch.no_grad()
def get_first_answer_topk_logits_partA(
    lm_model,
    connector,
    item,
    tokenizer,
    device: torch.device,
    top_k: int = 5,
):
    """
    Computes top-k logits at the first answer-generation position.
    Prompt:
        [BOS, visual tokens, question]
    We inspect logits at the final prompt position.
    """
    lm_model.eval()
    connector.eval()

    question = item["question"]

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    q_enc = tokenizer(
        [question],
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=48,
    )

    tokenizer.padding_side = old_padding_side

    clip_features = item["clip_features"].unsqueeze(0).to(device)
    q_ids = q_enc["input_ids"].to(device)

    input_embeddings = get_lm_input_embeddings(lm_model)

    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
    bos_ids = torch.full((1, 1), bos_id, dtype=torch.long, device=device)

    bos_embeds = input_embeddings(bos_ids)

    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        visual_embeds = connector(clip_features)
        visual_embeds = visual_embeds.to(dtype=next(lm_model.parameters()).dtype)

    q_embeds = input_embeddings(q_ids)

    inputs_embeds = torch.cat([bos_embeds, visual_embeds, q_embeds], dim=1)
    attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)

    outputs = lm_model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        use_cache=False,
    )

    logits = outputs.logits[:, -1, :].float().squeeze(0)
    top_vals, top_ids = torch.topk(logits, k=top_k)

    top_rows = []

    for rank, (tok_id, logit_val) in enumerate(zip(top_ids.tolist(), top_vals.tolist()), start=1):
        token_text = tokenizer.decode([tok_id], skip_special_tokens=False)
        top_rows.append({
            "rank": rank,
            "token_id": tok_id,
            "token": repr(token_text),
            "logit": float(logit_val),
        })

    return top_rows


def pick_qualitative_examples(phase_eval_df: pd.DataFrame):
    """
    Select:
        2 correct-easy: binary/coarse templates
        2 correct-hard: object_recognition
        2 failures: incorrect predictions
    """
    df = phase_eval_df.copy()

    easy_templates = ["is_animal", "can_fly", "vehicle_or_living", "class_presence"]

    correct_easy = df[
        (df["correct"] == True) &
        (df["template_id"].isin(easy_templates))
    ].head(2)

    correct_hard = df[
        (df["correct"] == True) &
        (df["template_id"] == "object_recognition")
    ].head(2)

    failures = df[
        df["correct"] == False
    ].head(2)

    selected = pd.concat([correct_easy, correct_hard, failures], axis=0)

    # If there are fewer than 2 failures, fill remaining with more hard examples.
    if len(selected) < 6:
        needed = 6 - len(selected)
        filler = df[
            (df["correct"] == True) &
            (~df.index.isin(selected.index))
        ].head(needed)
        selected = pd.concat([selected, filler], axis=0)

    selected = selected.head(6).reset_index(drop=True)
    return selected


qual_selected_A = pick_qualitative_examples(phase2_full_eval_A["df"])

qualitative_rows_A = []

print("Selected qualitative examples:")
print("=" * 100)

for q_idx, row in qual_selected_A.iterrows():
    dataset_i = int(row["i"])
    item = val_vqa_dataset_A[dataset_i]

    top5 = get_first_answer_topk_logits_partA(
        lm_model=lm_eval_A,
        connector=connector_eval_A,
        item=item,
        tokenizer=tokenizer,
        device=DEVICE,
        top_k=5,
    )

    category = (
        "correct-easy" if row["correct"] and row["template_id"] != "object_recognition"
        else "correct-hard" if row["correct"] and row["template_id"] == "object_recognition"
        else "failure"
    )

    print(f"\nExample {q_idx + 1}: {category}")
    print("-" * 100)
    print("Dataset index:", dataset_i)
    print("Class:", row["class_name"])
    print("Template:", row["template_id"])
    print("Question:", row["question"])
    print("Gold:", row["gold"])
    print("Raw prediction:", row["raw_pred"])
    print("Normalized prediction:", row["norm_pred"])
    print("Correct:", row["correct"])
    print("Top-5 logits at first answer token:")
    display(pd.DataFrame(top5))

    qualitative_rows_A.append({
        "category": category,
        "dataset_i": dataset_i,
        "class_name": row["class_name"],
        "template_id": row["template_id"],
        "question": row["question"],
        "gold": row["gold"],
        "raw_pred": row["raw_pred"],
        "norm_pred": row["norm_pred"],
        "correct": bool(row["correct"]),
        "top5": top5,
    })

raining time, VRAM, trainable-parameter summary

In [ ]:

resource_summary_A = pd.DataFrame([
    {
        "phase": "Phase 0: baseline eval",
        "trainable_params": 0,
        "trainable_percent_notes": "0%; no training",
        "train_time_min": 0.0,
        "peak_vram_gb": None,
        "notes": f"PPL0={PPL0:.4f}; majority/text-only baselines.",
    },
    {
        "phase": "Phase 1: connector init",
        "trainable_params": 1_660_800,
        "trainable_percent_notes": "connector only",
        "train_time_min": 0.67,
        "peak_vram_gb": 6.908,
        "notes": "Frozen CLIP + frozen LM; trained MLP connector on captions.",
    },
    {
        "phase": "Phase 2: VQA + replay",
        "trainable_params": 4_937_600,
        "trainable_percent_notes": "LoRA 0.8975% of LM; 1.3463% incl. connector",
        "train_time_min": 25.53,
        "peak_vram_gb": 5.42,
        "notes": "Full 50k VQA pairs with λ=0.2 Alpaca replay.",
    },
    {
        "phase": "Phase 3: VQA only",
        "trainable_params": 4_937_600,
        "trainable_percent_notes": "same LoRA + connector",
        "train_time_min": 3.42,
        "peak_vram_gb": 3.912,
        "notes": "10k VQA pairs, λ=0, lr=2e-4.",
    },
])

display(resource_summary_A)

In [ ]:
partA_eval_dir = os.path.join(OUTPUTS_DIR, "partA_final_A_C4")
os.makedirs(partA_eval_dir, exist_ok=True)

phase_summary_A.to_csv(
    os.path.join(partA_eval_dir, "phase0_to_phase3_accuracy_R_summary.csv"),
    index=False,
)

resource_summary_A.to_csv(
    os.path.join(partA_eval_dir, "phase_resource_summary.csv"),
    index=False,
)

with open(os.path.join(partA_eval_dir, "qualitative_examples_top5.json"), "w") as f:
    json.dump(qualitative_rows_A, f, indent=2)

with open(os.path.join(partA_eval_dir, "partA_C4_summary.json"), "w") as f:
    json.dump(
        {
            "phase2_full_val_acc": float(phase2_full_overall_acc),
            "phase2_PPL": float(phase2_final_PPL),
            "phase2_R": float(phase2_final_R),
            "phase3_full_val_acc": float(phase3_full_eval_A["overall_acc"]),
            "phase3_PPL": float(phase3_PPL),
            "phase3_R": float(phase3_R),
            "majority_global_acc": float(majority_baselines_A["global_majority_acc"]),
            "text_only_1000_acc": float(text_only_eval_A["overall_acc"]),
            "phase0_to_phase3_summary": phase_summary_A.to_dict(orient="records"),
            "resource_summary": resource_summary_A.to_dict(orient="records"),
        },
        f,
        indent=2,
    )

print("Saved final A-C4 outputs to:")
print(partA_eval_dir)
print("Files:")
print(os.listdir(partA_eval_dir))

Fixed 200-example subset for modality-gap analysis

In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

MODALITY_GAP_SEED = 42
MODALITY_GAP_N = 200

rng = np.random.default_rng(MODALITY_GAP_SEED)

# Select 200 fixed VQA validation examples.
# Each example gives us:
#   - one image local_id
#   - one question
mg_vqa_indices_A = rng.choice(
    len(val_vqa_dataset_A),
    size=MODALITY_GAP_N,
    replace=False,
).tolist()

mg_records_A = [val_vqa_dataset_A[i] for i in mg_vqa_indices_A]

mg_local_ids_A = [item["local_id"] for item in mg_records_A]
mg_questions_A = [item["question"] for item in mg_records_A]
mg_classes_A = [item["class_name"] for item in mg_records_A]
mg_templates_A = [item["template_id"] for item in mg_records_A]

mg_clip_features_A = torch.stack(
    [val_clip_features_A[local_id] for local_id in mg_local_ids_A],
    dim=0,
)

print("Fixed modality-gap examples:", len(mg_records_A))
print("mg_clip_features_A:", mg_clip_features_A.shape, mg_clip_features_A.dtype)

print("\nTemplate counts:")
print(Counter(mg_templates_A))

print("\nClass counts:")
print(Counter(mg_classes_A))

assert mg_clip_features_A.shape == (MODALITY_GAP_N, 49, 768)

print("\nFixed modality-gap subset created.")

Modality-gap metric utilities

In [ ]:
@torch.no_grad()
def mean_pool_text_question_embeddings(
    lm_model,
    questions: List[str],
    tokenizer,
    device: torch.device,
    batch_size: int = 32,
    max_length: int = 48,
) -> torch.Tensor:
    """
    Returns one mean-pooled text embedding per question:
        [N, dLM]

    Uses the LM input embedding table.
    """
    lm_model.eval()
    input_embeddings = get_lm_input_embeddings(lm_model)

    all_text_reps = []

    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    for start in range(0, len(questions), batch_size):
        batch_questions = questions[start:start + batch_size]

        enc = tokenizer(
            batch_questions,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
            add_special_tokens=False,
        )

        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        text_embeds = input_embeddings(input_ids).float()

        mask = attention_mask.unsqueeze(-1).float()
        summed = (text_embeds * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp_min(1.0)

        pooled = summed / denom
        all_text_reps.append(pooled.detach().cpu())

    tokenizer.padding_side = old_padding_side

    return torch.cat(all_text_reps, dim=0)


@torch.no_grad()
def mean_pool_visual_embeddings(
    connector,
    clip_features: torch.Tensor,
    device: torch.device,
    batch_size: int = 64,
) -> torch.Tensor:
    """
    Returns one mean-pooled visual embedding per image:
        [N, dLM]
    """
    connector.eval()

    all_visual_reps = []

    for start in range(0, clip_features.shape[0], batch_size):
        batch_features = clip_features[start:start + batch_size].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            visual_tokens = connector(batch_features)

        visual_pooled = visual_tokens.float().mean(dim=1)
        all_visual_reps.append(visual_pooled.detach().cpu())

    return torch.cat(all_visual_reps, dim=0)


def average_off_diagonal_cosine(x: torch.Tensor) -> float:
    """
    Average pairwise cosine similarity excluding diagonal.
    x: [N, D], assumed unnormalised or normalised.
    """
    x = F.normalize(x.float(), dim=-1)
    sim = x @ x.T

    n = sim.shape[0]
    off_diag = sim[~torch.eye(n, dtype=torch.bool)]

    return off_diag.mean().item()


def average_cross_modal_cosine(v: torch.Tensor, t: torch.Tensor) -> float:
    """
    Average cosine similarity across all visual-text pairs.
    """
    v = F.normalize(v.float(), dim=-1)
    t = F.normalize(t.float(), dim=-1)

    sim = v @ t.T
    return sim.mean().item()


def compute_modality_gap_metrics(
    visual_reps: torch.Tensor,
    text_reps: torch.Tensor,
) -> Dict[str, float]:
    """
    Computes:
        MG = || mean(normalized visual) - mean(normalized text) ||_2
        within visual cosine
        within text cosine
        cross-modal cosine
        visual/text norm ratio
    """
    visual_reps = visual_reps.float()
    text_reps = text_reps.float()

    v_normed = F.normalize(visual_reps, dim=-1)
    t_normed = F.normalize(text_reps, dim=-1)

    v_bar = v_normed.mean(dim=0)
    t_bar = t_normed.mean(dim=0)

    MG = torch.norm(v_bar - t_bar, p=2).item()

    within_visual = average_off_diagonal_cosine(visual_reps)
    within_text = average_off_diagonal_cosine(text_reps)
    cross_modal = average_cross_modal_cosine(visual_reps, text_reps)

    visual_norm_mean = visual_reps.norm(dim=-1).mean().item()
    text_norm_mean = text_reps.norm(dim=-1).mean().item()
    norm_ratio = visual_norm_mean / max(text_norm_mean, 1e-8)

    return {
        "MG": MG,
        "within_visual_cos": within_visual,
        "within_text_cos": within_text,
        "cross_modal_cos": cross_modal,
        "visual_norm_mean": visual_norm_mean,
        "text_norm_mean": text_norm_mean,
        "norm_ratio": norm_ratio,
    }


@torch.no_grad()
def compute_phase_modality_gap(
    phase_name: str,
    lm_model,
    connector,
    clip_features: torch.Tensor,
    questions: List[str],
    tokenizer,
    device: torch.device,
) -> Tuple[Dict[str, float], torch.Tensor, torch.Tensor]:
    """
    Computes modality gap metrics for one checkpoint.
    """
    visual_reps = mean_pool_visual_embeddings(
        connector=connector,
        clip_features=clip_features,
        device=device,
        batch_size=64,
    )

    text_reps = mean_pool_text_question_embeddings(
        lm_model=lm_model,
        questions=questions,
        tokenizer=tokenizer,
        device=device,
        batch_size=32,
        max_length=48,
    )

    metrics = compute_modality_gap_metrics(
        visual_reps=visual_reps,
        text_reps=text_reps,
    )

    metrics["phase"] = phase_name

    return metrics, visual_reps, text_reps

Checkpoint loaders for phases 0–3

In [ ]:
def load_base_lm_for_embedding_eval():
    """
    Load frozen base LM for input-embedding comparisons.
    LoRA is not needed for modality gap because we compare connector
    outputs to LM input-token embeddings.
    """
    model = AutoModelForCausalLM.from_pretrained(
        cfg.lm_name,
        torch_dtype=cfg.dtype,
        device_map=None,
    ).to(DEVICE)

    model.config.use_cache = False
    set_requires_grad(model, False)
    model.eval()

    return model


def load_connector_from_state_dict_path(path: str):
    payload = torch.load(path, map_location=DEVICE)

    connector = MLPConnector(
        in_dim=partA_cfg.clip_hidden_size,
        hidden_dim=partA_cfg.lm_hidden_size,
        out_dim=partA_cfg.lm_hidden_size,
    ).to(DEVICE)

    connector.load_state_dict(payload["connector_state_dict"])
    connector.eval()
    set_requires_grad(connector, False)

    return connector, payload


def load_connector_from_payload_path(path: str):
    return load_connector_from_state_dict_path(path)


# Phase 0: random untrained connector.
# This is a pre-alignment reference, not a trained VQA model.
seed_everything(42)
connector_phase0_A = MLPConnector(
    in_dim=partA_cfg.clip_hidden_size,
    hidden_dim=partA_cfg.lm_hidden_size,
    out_dim=partA_cfg.lm_hidden_size,
).to(DEVICE)
connector_phase0_A.eval()
set_requires_grad(connector_phase0_A, False)

# Phase 1: soft-rescaled connector
connector_phase1_A, phase1_payload_for_mg = load_connector_from_payload_path(
    phase1_soft_ckpt_path_A
)

# Phase 2: full baseline connector
connector_phase2_A, phase2_payload_for_mg = load_connector_from_payload_path(
    phase2_full_connector_path_A
)

# Phase 3: VQA-only connector
connector_phase3_A, phase3_payload_for_mg = load_connector_from_payload_path(
    phase3_connector_path_A
)

lm_embed_eval_A = load_base_lm_for_embedding_eval()

print("Loaded base LM and phase connectors for modality-gap analysis.")
print("Phase 1 stored rnorm:", phase1_payload_for_mg.get("rnorm_phase1_soft_rescaled", None))
print("Phase 2 payload keys:", phase2_payload_for_mg.keys())
print("Phase 3 payload keys:", phase3_payload_for_mg.keys())

Compute modality gap across phases 0–3

In [ ]:
phase_connectors_for_mg_A = {
    "Phase 0: random connector": connector_phase0_A,
    "Phase 1: caption init": connector_phase1_A,
    "Phase 2: VQA + replay": connector_phase2_A,
    "Phase 3: VQA only": connector_phase3_A,
}

mg_metrics_rows_A = []
mg_representations_A = {}

for phase_name, connector in phase_connectors_for_mg_A.items():
    print("\n" + "=" * 80)
    print("Computing modality gap for:", phase_name)
    print("=" * 80)

    metrics, visual_reps, text_reps = compute_phase_modality_gap(
        phase_name=phase_name,
        lm_model=lm_embed_eval_A,
        connector=connector,
        clip_features=mg_clip_features_A,
        questions=mg_questions_A,
        tokenizer=tokenizer,
        device=DEVICE,
    )

    mg_metrics_rows_A.append(metrics)

    mg_representations_A[phase_name] = {
        "visual_reps": visual_reps,
        "text_reps": text_reps,
    }

    print(metrics)

mg_metrics_df_A = pd.DataFrame(mg_metrics_rows_A)

# Reorder columns
mg_metrics_df_A = mg_metrics_df_A[
    [
        "phase",
        "MG",
        "cross_modal_cos",
        "within_visual_cos",
        "within_text_cos",
        "visual_norm_mean",
        "text_norm_mean",
        "norm_ratio",
    ]
]

display(mg_metrics_df_A)

Plot modality gap metrics

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(mg_metrics_df_A["phase"], mg_metrics_df_A["MG"], marker="o")
plt.xticks(rotation=25, ha="right")
plt.ylabel("MG = ||mean(v) - mean(t)||₂")
plt.title("Part A Modality Gap Across Phases")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(mg_metrics_df_A["phase"], mg_metrics_df_A["cross_modal_cos"], marker="o", label="cross-modal")
plt.plot(mg_metrics_df_A["phase"], mg_metrics_df_A["within_visual_cos"], marker="o", label="within-visual")
plt.plot(mg_metrics_df_A["phase"], mg_metrics_df_A["within_text_cos"], marker="o", label="within-text")
plt.xticks(rotation=25, ha="right")
plt.ylabel("Average cosine similarity")
plt.title("Part A Within/Cross-Modal Cosine Similarities")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(mg_metrics_df_A["phase"], mg_metrics_df_A["norm_ratio"], marker="o")
plt.axhline(1.0, linestyle="--", label="equal visual/text norm")
plt.xticks(rotation=25, ha="right")
plt.ylabel("Visual/Text norm ratio")
plt.title("Part A Visual/Text Norm Ratio Across Phases")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

UMAP visualization across phases

In [ ]:
import umap

def plot_umap_for_phase(
    phase_name: str,
    visual_reps: torch.Tensor,
    text_reps: torch.Tensor,
):
    """
    UMAP projection of visual and text representations.
    """
    v = F.normalize(visual_reps.float(), dim=-1).numpy()
    t = F.normalize(text_reps.float(), dim=-1).numpy()

    X = np.concatenate([v, t], axis=0)
    labels = np.array(["visual"] * len(v) + ["text"] * len(t))

    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42,
    )

    X_2d = reducer.fit_transform(X)

    plt.figure(figsize=(6, 5))

    visual_mask = labels == "visual"
    text_mask = labels == "text"

    plt.scatter(
        X_2d[visual_mask, 0],
        X_2d[visual_mask, 1],
        alpha=0.75,
        label="visual",
        s=18,
    )

    plt.scatter(
        X_2d[text_mask, 0],
        X_2d[text_mask, 1],
        alpha=0.75,
        label="text",
        s=18,
    )

    plt.title(f"UMAP: {phase_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


for phase_name, reps in mg_representations_A.items():
    plot_umap_for_phase(
        phase_name=phase_name,
        visual_reps=reps["visual_reps"],
        text_reps=reps["text_reps"],
    )

In [ ]:
mg_output_dir_A = os.path.join(OUTPUTS_DIR, "partA_modality_gap_A_C5")
os.makedirs(mg_output_dir_A, exist_ok=True)

mg_metrics_df_A.to_csv(
    os.path.join(mg_output_dir_A, "modality_gap_phase0_to_phase3.csv"),
    index=False,
)

torch.save(
    {
        "mg_metrics_df": mg_metrics_df_A,
        "mg_vqa_indices": mg_vqa_indices_A,
        "mg_local_ids": mg_local_ids_A,
        "mg_questions": mg_questions_A,
        "mg_classes": mg_classes_A,
        "mg_templates": mg_templates_A,
        "representations": mg_representations_A,
    },
    os.path.join(mg_output_dir_A, "modality_gap_representations.pt"),
)

print("Saved A-C5 modality-gap outputs to:")
print(mg_output_dir_A)
print("Files:")
print(os.listdir(mg_output_dir_A))

Lnorm training loop

In [ ]:

# Phase 2 + Lnorm compact rerun

# ============================================================
# PATCH: Fixed dtype-safe Lnorm computation
# ============================================================

def compute_lnorm_for_vqa_batch(
    lm_model,
    connector,
    clip_features,
    question_input_ids,
    question_attention_mask,
    answer_input_ids,
    answer_attention_mask,
    device,
):
    """
    Lnorm = (E||V_i|| - E||T_j||)^2

    Fix:
    Cached CLIP features are fp16, while connector weights may be fp32.
    We cast CLIP features to the connector parameter dtype before connector forward.
    """
    # Detect connector dtype safely
    connector_dtype = next(connector.parameters()).dtype

    clip_features = clip_features.to(
        device=device,
        dtype=connector_dtype,
        non_blocking=True,
    )

    question_input_ids = question_input_ids.to(device, non_blocking=True)
    question_attention_mask = question_attention_mask.to(device, non_blocking=True)
    answer_input_ids = answer_input_ids.to(device, non_blocking=True)
    answer_attention_mask = answer_attention_mask.to(device, non_blocking=True)

    input_embeddings = get_lm_input_embeddings(lm_model)

    # Connector output can be fp32/fp16 depending on connector dtype.
    # We convert to float32 only for stable norm computation.
    visual_tokens = connector(clip_features).float()

    # Embedding outputs may be fp16, convert to float32 for norms.
    q_embeds = input_embeddings(question_input_ids).float()
    a_embeds = input_embeddings(answer_input_ids).float()

    q_mask = question_attention_mask.bool()
    a_mask = answer_attention_mask.bool()

    text_embeds = torch.cat(
        [
            q_embeds[q_mask],
            a_embeds[a_mask],
        ],
        dim=0,
    )

    visual_norm_mean = visual_tokens.norm(dim=-1).mean()
    text_norm_mean = text_embeds.norm(dim=-1).mean()

    lnorm = (visual_norm_mean - text_norm_mean).pow(2)

    return lnorm, visual_norm_mean.detach(), text_norm_mean.detach()


def train_phase2_with_lnorm_compact(
    lm_model,
    connector,
    vqa_loader,
    replay_loader,
    val_dataset,
    optimizer,
    scheduler,
    device,
    lambda_replay: float = 0.2,
    lnorm_weight: float = 1e-2,
    grad_accum_steps: int = 4,
    max_train_steps: int = 600,
    log_every: int = 100,
    eval_every: int = 600,
):
    lm_model.train()
    connector.train()

    scaler = torch.amp.GradScaler("cuda")
    replay_iter = infinite_loader(replay_loader)

    history = {
        "step": [],
        "vqa_loss": [],
        "replay_loss": [],
        "lnorm_loss": [],
        "mixed_loss": [],
        "visual_norm": [],
        "text_norm": [],
        "quick_vqa_acc": [],
        "ppl_replay": [],
        "R": [],
    }

    optimizer.zero_grad(set_to_none=True)
    reset_peak_memory()
    start_time = time.time()

    pbar = tqdm(total=max_train_steps, desc="Phase 2 + Lnorm compact")

    global_step = 0

    while global_step < max_train_steps:
        for vqa_batch in vqa_loader:
            if global_step >= max_train_steps:
                break

            replay_batch = next(replay_iter)

            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                inputs_embeds, attention_mask, labels = build_phase2_vqa_inputs_and_labels(
                    lm_model=lm_model,
                    connector=connector,
                    clip_features=vqa_batch["clip_features"],
                    question_input_ids=vqa_batch["question_input_ids"],
                    question_attention_mask=vqa_batch["question_attention_mask"],
                    answer_input_ids=vqa_batch["answer_input_ids"],
                    answer_attention_mask=vqa_batch["answer_attention_mask"],
                    device=device,
                )

                vqa_outputs = lm_model(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    labels=labels,
                    use_cache=False,
                )

                vqa_loss = vqa_outputs.loss

                replay_input_ids = replay_batch["input_ids"].to(device, non_blocking=True)
                replay_attention_mask = replay_batch["attention_mask"].to(device, non_blocking=True)
                replay_labels = replay_batch["labels"].to(device, non_blocking=True)

                replay_outputs = lm_model(
                    input_ids=replay_input_ids,
                    attention_mask=replay_attention_mask,
                    labels=replay_labels,
                    use_cache=False,
                )

                replay_loss = replay_outputs.loss

            # Compute Lnorm outside autocast for stable norm values.
            lnorm_loss, vnorm, tnorm = compute_lnorm_for_vqa_batch(
                lm_model=lm_model,
                connector=connector,
                clip_features=vqa_batch["clip_features"],
                question_input_ids=vqa_batch["question_input_ids"],
                question_attention_mask=vqa_batch["question_attention_mask"],
                answer_input_ids=vqa_batch["answer_input_ids"],
                answer_attention_mask=vqa_batch["answer_attention_mask"],
                device=device,
            )

            mixed_loss = vqa_loss + lambda_replay * replay_loss + lnorm_weight * lnorm_loss
            loss_for_backward = mixed_loss / grad_accum_steps

            scaler.scale(loss_for_backward).backward()

            if (global_step + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)

                trainable_params = [
                    p for p in list(lm_model.parameters()) + list(connector.parameters())
                    if p.requires_grad and p.grad is not None
                ]

                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if scheduler is not None:
                    scheduler.step()

            global_step += 1
            pbar.update(1)

            if global_step % log_every == 0:
                print(
                    f"step={global_step:04d} | "
                    f"vqa={vqa_loss.item():.4f} | "
                    f"replay={replay_loss.item():.4f} | "
                    f"lnorm={lnorm_loss.item():.4f} | "
                    f"mixed={mixed_loss.item():.4f} | "
                    f"vnorm={vnorm.item():.3f} | "
                    f"tnorm={tnorm.item():.3f} | "
                    f"mem={get_gpu_memory_gb()}"
                )

                history["step"].append(global_step)
                history["vqa_loss"].append(float(vqa_loss.detach().cpu()))
                history["replay_loss"].append(float(replay_loss.detach().cpu()))
                history["lnorm_loss"].append(float(lnorm_loss.detach().cpu()))
                history["mixed_loss"].append(float(mixed_loss.detach().cpu()))
                history["visual_norm"].append(float(vnorm.cpu()))
                history["text_norm"].append(float(tnorm.cpu()))

            if global_step % eval_every == 0:
                quick_acc, _ = evaluate_vqa_partA_small(
                    lm_model=lm_model,
                    connector=connector,
                    dataset=val_dataset,
                    max_examples=200,
                )

                ppl_fine = compute_perplexity(
                    model=lm_model,
                    tokenizer=tokenizer,
                    texts=ppl0_texts,
                    device=device,
                    batch_size=4,
                    max_length=cfg.max_text_length,
                    desc=f"Lnorm replay PPL step {global_step}",
                )

                R = ppl_fine / PPL0

                print(
                    f"\n[LNORM EVAL step={global_step}] "
                    f"quick_vqa_acc={quick_acc:.4f} | "
                    f"PPL={ppl_fine:.4f} | "
                    f"R={R:.4f}\n"
                )

                history["quick_vqa_acc"].append((global_step, quick_acc))
                history["ppl_replay"].append((global_step, ppl_fine))
                history["R"].append((global_step, R))

                lm_model.train()
                connector.train()

    pbar.close()

    elapsed = time.time() - start_time

    print("\nPhase 2 + Lnorm compact training complete.")
    print("Elapsed minutes:", round(elapsed / 60, 2))
    print("Peak GPU memory GB:", get_gpu_memory_gb()["max_allocated"])

    return history

Run compact Phase 2 + Lnorm

In [ ]:
lm_lnorm_A, connector_lnorm_A = reset_partA_phase2_model_for_ablation()

lnorm_max_steps = 600
lnorm_grad_accum = 4
lnorm_optimizer_steps = math.ceil(lnorm_max_steps / lnorm_grad_accum)

lnorm_trainable_params = [
    p for p in list(lm_lnorm_A.parameters()) + list(connector_lnorm_A.parameters())
    if p.requires_grad
]

lnorm_optimizer_A = torch.optim.AdamW(
    lnorm_trainable_params,
    lr=5e-4,
    weight_decay=0.01,
)

lnorm_scheduler_A = torch.optim.lr_scheduler.OneCycleLR(
    lnorm_optimizer_A,
    max_lr=5e-4,
    total_steps=lnorm_optimizer_steps,
    pct_start=0.10,
    anneal_strategy="cos",
)

phase2_lnorm_history_A = train_phase2_with_lnorm_compact(
    lm_model=lm_lnorm_A,
    connector=connector_lnorm_A,
    vqa_loader=train_vqa_loader_A,
    replay_loader=alpaca_replay_loader,
    val_dataset=val_vqa_dataset_A,
    optimizer=lnorm_optimizer_A,
    scheduler=lnorm_scheduler_A,
    device=DEVICE,
    lambda_replay=0.2,
    lnorm_weight=1e-2,
    grad_accum_steps=lnorm_grad_accum,
    max_train_steps=lnorm_max_steps,
    log_every=100,
    eval_every=600,
)

lnorm_final_acc = phase2_lnorm_history_A["quick_vqa_acc"][-1][1]
lnorm_final_ppl = phase2_lnorm_history_A["ppl_replay"][-1][1]
lnorm_final_R = phase2_lnorm_history_A["R"][-1][1]

print("Lnorm compact result:")
print({
    "quick_vqa_acc": lnorm_final_acc,
    "ppl": lnorm_final_ppl,
    "R": lnorm_final_R,
})

Compute modality gap for Lnorm model and compare

In [ ]:
lnorm_metrics, lnorm_visual_reps, lnorm_text_reps = compute_phase_modality_gap(
    phase_name="Phase 2 + Lnorm compact",
    lm_model=lm_embed_eval_A,
    connector=connector_lnorm_A,
    clip_features=mg_clip_features_A,
    questions=mg_questions_A,
    tokenizer=tokenizer,
    device=DEVICE,
)

baseline_phase2_mg = mg_metrics_df_A[
    mg_metrics_df_A["phase"] == "Phase 2: VQA + replay"
].iloc[0].to_dict()

lnorm_compare_A = pd.DataFrame([
    {
        "condition": "Phase 2 baseline, λ=0.2",
        "steps": 600,
        "quick_vqa_acc": phase2_history_A["quick_vqa_acc"][-1][1],
        "PPL": phase2_history_A["ppl_replay"][-1][1],
        "R": phase2_history_A["R"][-1][1],
        "MG": baseline_phase2_mg["MG"],
        "cross_modal_cos": baseline_phase2_mg["cross_modal_cos"],
        "norm_ratio": baseline_phase2_mg["norm_ratio"],
    },
    {
        "condition": "Phase 2 + Lnorm, λ=0.2",
        "steps": 600,
        "quick_vqa_acc": lnorm_final_acc,
        "PPL": lnorm_final_ppl,
        "R": lnorm_final_R,
        "MG": lnorm_metrics["MG"],
        "cross_modal_cos": lnorm_metrics["cross_modal_cos"],
        "norm_ratio": lnorm_metrics["norm_ratio"],
    },
])

display(lnorm_compare_A)

In [ ]:
lnorm_output_dir_A = os.path.join(OUTPUTS_DIR, "partA_lnorm_diagnostic_A_C5")
os.makedirs(lnorm_output_dir_A, exist_ok=True)

lnorm_compare_A.to_csv(
    os.path.join(lnorm_output_dir_A, "phase2_baseline_vs_lnorm_600steps.csv"),
    index=False,
)

torch.save(
    {
        "history": phase2_lnorm_history_A,
        "metrics": lnorm_metrics,
        "visual_reps": lnorm_visual_reps,
        "text_reps": lnorm_text_reps,
    },
    os.path.join(lnorm_output_dir_A, "phase2_lnorm_diagnostic.pt"),
)

print("Saved Lnorm diagnostic outputs to:")
print(lnorm_output_dir_A)
print(os.listdir(lnorm_output_dir_A))

Ablation: LoRA Rank Sweep

Reset model with arbitrary LoRA rank

In [ ]:
def reset_partA_phase2_model_with_lora_rank(lora_rank: int):
    """
    Reloads:
        clean frozen SmolLM2
        soft-rescaled Phase 1 connector
        fresh LoRA with chosen rank
    """
    global lm_rank_tmp, connector_rank_tmp

    gc.collect()
    torch.cuda.empty_cache()

    lm_rank_tmp = AutoModelForCausalLM.from_pretrained(
        cfg.lm_name,
        torch_dtype=cfg.dtype,
        device_map=None,
    ).to(DEVICE)

    lm_rank_tmp.config.use_cache = False
    set_requires_grad(lm_rank_tmp, False)
    lm_rank_tmp.eval()

    connector_rank_tmp = MLPConnector(
        in_dim=partA_cfg.clip_hidden_size,
        hidden_dim=partA_cfg.lm_hidden_size,
        out_dim=partA_cfg.lm_hidden_size,
    ).to(DEVICE)

    phase1_payload = torch.load(phase1_soft_ckpt_path_A, map_location=DEVICE)
    connector_rank_tmp.load_state_dict(phase1_payload["connector_state_dict"])
    connector_rank_tmp.train()
    set_requires_grad(connector_rank_tmp, True)

    lora_config_rank = LoraConfig(
        r=lora_rank,
        lora_alpha=2 * lora_rank,  # keep alpha/r roughly stable; r=16 gives alpha=32
        target_modules=list(cfg.lora_targets),
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    lm_rank_lora = get_peft_model(lm_rank_tmp, lora_config_rank)
    lm_rank_lora.to(DEVICE)
    lm_rank_lora.train()

    return lm_rank_lora, connector_rank_tmp

Run compact LoRA rank sweep

In [ ]:
lora_rank_values_A = [2, 4, 16, 32]

lora_rank_ablation_results_A = {}

for rank in lora_rank_values_A:
    print("\n" + "=" * 100)
    print(f"Running LoRA rank ablation: r = {rank}")
    print("=" * 100)

    # Use existing 600-step r=16 pilot if available to avoid rerunning it.
    if rank == 16 and "phase2_history_A" in globals():
        print("Using existing 600-step r=16 Phase 2 pilot result.")

        lora_rank_ablation_results_A[rank] = {
            "rank": rank,
            "lora_alpha": 32,
            "steps": 600,
            "history": phase2_history_A,
            "quick_vqa_acc": phase2_history_A["quick_vqa_acc"][-1][1],
            "ppl": phase2_history_A["ppl_replay"][-1][1],
            "R": phase2_history_A["R"][-1][1],
            "trainable_lora_params": 3_276_800,
        }

        continue

    lm_rank_A, connector_rank_A = reset_partA_phase2_model_with_lora_rank(rank)

    trainable_lora_params = sum(
        p.numel() for p in lm_rank_A.parameters()
        if p.requires_grad
    )

    total_lora_model_params = sum(p.numel() for p in lm_rank_A.parameters())

    print_parameter_summary(lm_rank_A, f"SmolLM2 LoRA rank {rank}")
    print("Trainable LoRA params:", trainable_lora_params)

    max_steps = 600
    grad_accum = 4
    optimizer_steps = math.ceil(max_steps / grad_accum)

    trainable_params = [
        p for p in list(lm_rank_A.parameters()) + list(connector_rank_A.parameters())
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=5e-4,
        weight_decay=0.01,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=5e-4,
        total_steps=optimizer_steps,
        pct_start=0.10,
        anneal_strategy="cos",
    )

    history = train_phase2_vqa_with_replay(
        lm_model=lm_rank_A,
        connector=connector_rank_A,
        vqa_loader=train_vqa_loader_A,
        replay_loader=alpaca_replay_loader,
        val_dataset=val_vqa_dataset_A,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        lambda_replay=0.2,
        num_epochs=1,
        grad_accum_steps=grad_accum,
        log_every=100,
        eval_every=600,
        max_train_steps=max_steps,
    )

    final_acc = history["quick_vqa_acc"][-1][1]
    final_ppl = history["ppl_replay"][-1][1]
    final_R = history["R"][-1][1]

    lora_rank_ablation_results_A[rank] = {
        "rank": rank,
        "lora_alpha": 2 * rank,
        "steps": max_steps,
        "history": history,
        "quick_vqa_acc": final_acc,
        "ppl": final_ppl,
        "R": final_R,
        "trainable_lora_params": trainable_lora_params,
        "trainable_lora_percent": 100 * trainable_lora_params / total_lora_model_params,
    }

    print("\nRank ablation result:")
    print({
        "rank": rank,
        "quick_vqa_acc": final_acc,
        "ppl": final_ppl,
        "R": final_R,
        "trainable_lora_params": trainable_lora_params,
    })

Build and plot LoRA rank ablation table

In [ ]:
rank_rows = []

for rank in lora_rank_values_A:
    result = lora_rank_ablation_results_A[rank]

    rank_rows.append({
        "lora_rank": rank,
        "lora_alpha": result["lora_alpha"],
        "steps": result["steps"],
        "trainable_lora_params": result["trainable_lora_params"],
        "quick_vqa_acc": result["quick_vqa_acc"],
        "ppl": result["ppl"],
        "R": result["R"],
    })

lora_rank_ablation_df_A = pd.DataFrame(rank_rows)
display(lora_rank_ablation_df_A)

plt.figure(figsize=(7, 4))
plt.plot(
    lora_rank_ablation_df_A["lora_rank"],
    lora_rank_ablation_df_A["quick_vqa_acc"],
    marker="o",
)
plt.xlabel("LoRA rank")
plt.ylabel("Quick VQA accuracy")
plt.title("Part A A-C6: LoRA Rank vs Quick VQA Accuracy")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(
    lora_rank_ablation_df_A["lora_rank"],
    lora_rank_ablation_df_A["R"],
    marker="o",
)
plt.axhline(1.0, linestyle="--", label="No forgetting threshold")
plt.xlabel("LoRA rank")
plt.ylabel("Forgetting ratio R")
plt.title("Part A A-C6: LoRA Rank vs Forgetting")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
a_c6_output_dir_A = os.path.join(OUTPUTS_DIR, "partA_A_C6_lora_rank_ablation")
os.makedirs(a_c6_output_dir_A, exist_ok=True)

lora_rank_ablation_df_A.to_csv(
    os.path.join(a_c6_output_dir_A, "lora_rank_ablation_600steps.csv"),
    index=False,
)

torch.save(
    lora_rank_ablation_results_A,
    os.path.join(a_c6_output_dir_A, "lora_rank_ablation_histories.pt"),
)

print("Saved A-C6 LoRA rank ablation outputs to:")
print(a_c6_output_dir_A)
print(os.listdir(a_c6_output_dir_A))

## Task A-C5: Modality Gap Analysis

We computed modality gap on a fixed set of 200 validation examples sampled with seed 42 and reused the same image/question pairs across all checkpoints. Visual representations were obtained by passing cached CLIP patch tokens through the connector and mean-pooling the 49 visual tokens. Text representations were obtained by mean-pooling the LM input embeddings of the corresponding questions.

The modality gap decreased after VQA training. The random connector had `MG = 1.2141`, Phase 1 caption initialization had `MG = 1.2305`, Phase 2 VQA + replay reduced it to `MG = 1.0916`, and Phase 3 VQA-only tuning reduced it further to `MG = 1.0668`. This suggests that supervised VQA alignment makes connector outputs geometrically more compatible with the LM/text embedding space.

However, the lowest modality gap did not correspond to the best VQA accuracy. Phase 3 had the lowest MG but slightly lower full validation accuracy than Phase 2. Therefore, MG is a useful diagnostic of visual/text representation geometry, but it is not a substitute for task accuracy.

The within-visual cosine also changed substantially. Phase 1 had very high within-visual cosine (`0.9934`), suggesting visual embeddings were highly collapsed. After VQA training, within-visual cosine dropped to `0.6563` in Phase 2 and `0.6069` in Phase 3, indicating that VQA supervision diversified the visual token representations.

We also tested an `Lnorm` regularizer:
`Lnorm = (E||V_i|| - E||T_j||)^2`.
This reduced the visual/text norm ratio from `4.33` to `1.76`, but it reduced quick VQA accuracy from `0.93` to `0.90` and increased MG from `1.0916` to `1.1382`. Thus, norm alignment helped control magnitude mismatch but did not improve alignment quality or downstream VQA performance in this compact run.

## Task A-C6: LoRA Rank Ablation

We ran a compact LoRA rank sweep with `r ∈ {2, 4, 16, 32}`. Each condition started from the same soft-rescaled Phase 1 connector and used the same Phase 2 mixed objective with `λ = 0.2`. Each model was trained for 600 VQA batches and evaluated on the same 200-example quick VQA set and 100 Alpaca replay examples.

The results show a clear adaptation-capacity trend. Rank 2 achieved `0.890` quick VQA accuracy with `R = 0.7920`. Rank 4 improved to `0.930` with `R = 0.7600`. Rank 16 kept quick VQA at `0.930` but improved replay perplexity, giving `R = 0.6335`. Rank 32 achieved the best result, with `0.965` quick VQA accuracy and `R = 0.6006`.

This suggests that, in this setup, higher LoRA rank improves both visual task adaptation and replay perplexity. The tradeoff is parameter cost: rank 2 uses only `409,600` LoRA parameters, rank 4 uses `819,200`, rank 16 uses `3,276,800`, and rank 32 uses `6,553,600`. The baseline rank 16 is a good compute-efficient choice, while rank 32 gives the best compact-run accuracy.